In [ ]:
# import os
# import cv2
# import numpy as np
# import pandas as pd
# from pathlib import Path
# from math import atan, pi
# from skimage.color import rgb2lab
# import mediapipe as mp

# # Monk Skin Tone (MST) reference hex codes (1 to 10)
# mst_palette = [
#     "#f6ede4", "#f3e7db", "#f7ead0", "#eadaba", "#d7bd96",
#     "#a07e56", "#825c43", "#604134", "#3a312a", "#292420"
# ]
# mst_lab_palette = [rgb2lab(np.array([[tuple(int(h[i:i+2], 16)/255 for i in (1, 3, 5))]])) for h in mst_palette]

# def compute_ita(lab):
#     L, a, b = lab[0, 0]
#     return atan((L - 50) / b) * 180 / pi

# def find_closest_mst(lab_color):
#     ita_input = compute_ita(lab_color)
#     distances = [abs(ita_input - compute_ita(ref)) for ref in mst_lab_palette]
#     return distances.index(min(distances)) + 1

# def extract_face_skin_color(image_path):
#     img = cv2.imread(image_path)
#     img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

#     mp_face = mp.solutions.face_mesh
#     with mp_face.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True) as face_mesh:
#         result = face_mesh.process(img_rgb)
#         if not result.multi_face_landmarks:
#             return None

#         h, w, _ = img.shape
#         points = result.multi_face_landmarks[0].landmark

#         # Use landmarks from cheeks and forehead (minimal occlusion regions)
#         idxs = [162, 389, 10, 109, 338]  # forehead + cheeks
#         pixels = []
#         for idx in idxs:
#             x = min(w - 1, max(0, int(points[idx].x * w)))
#             y = min(h - 1, max(0, int(points[idx].y * h)))
#             pixels.append(img_rgb[y, x])
#             cv2.circle(img, (x, y), 3, (0, 255, 0), -1)  # draw landmark point

#         mean_rgb = np.mean(pixels, axis=0) / 255
#         lab = rgb2lab([[mean_rgb]])
#         ita = compute_ita(lab)
#         mst = find_closest_mst(lab)

#         # Save debug image with landmarks drawn
#         debug_path = image_path.replace(".jpg", "_landmarks.jpg")
#         cv2.imwrite(debug_path, img)

#         return {
#             "r": int(mean_rgb[0] * 255),
#             "g": int(mean_rgb[1] * 255),
#             "b": int(mean_rgb[2] * 255),
#             "l": lab[0, 0, 0],
#             "a": lab[0, 0, 1],
#             "b_lab": lab[0, 0, 2],
#             "ita": ita,
#             "mst": mst
#         }

# def batch_process_images(img_dir, output_csv):
#     records = []
#     for path in Path(img_dir).rglob("*.jpg"):
#         result = extract_face_skin_color(str(path))
#         if result:
#             result["id"] = path.name
#             records.append(result)

#     df = pd.DataFrame(records)
#     df.to_csv(output_csv, index=False)
#     print(f"Saved skin tone data for {len(df)} faces to {output_csv}")

# # Example usage:
# for i in range(19):
#     subject = f'subject_{i}'
#     print(f"Processing {subject}")
#     batch_process_images(rf"MonkSkinToneDataset\mst-e_data\{subject}", rf"MonkSkinToneDataset\{subject}.csv")


Processing subject_0
Saved skin tone data for 38 faces to MonkSkinToneDataset\subject_0.csv
Processing subject_1


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from math import atan, pi
from skimage.color import rgb2lab
import mediapipe as mp
from ultralytics import YOLO

# Monk Skin Tone (MST) reference hex codes (1 to 10)
mst_palette = [
    "#f6ede4", "#f3e7db", "#f7ead0", "#eadaba", "#d7bd96",
    "#a07e56", "#825c43", "#604134", "#3a312a", "#292420"
]
mst_rgb_palette = [tuple(int(h[i:i+2], 16)/255 for i in (1, 3, 5)) for h in mst_palette]
mst_lab_palette = [rgb2lab(np.array([[rgb]])) for rgb in mst_rgb_palette]

actual_annotations = {
    'subject_0': 3, 'subject_1': 2, 'subject_2': 8, 'subject_3': 6,
    'subject_4': 9, 'subject_5': 7, 'subject_6': 5, 'subject_7': 4,
    'subject_8': 2, 'subject_9': 4, 'subject_10': 9, 'subject_11': 5,
    'subject_12': 10, 'subject_13': 2, 'subject_14': 6, 'subject_15': 3,
    'subject_16': 1, 'subject_17': 8, 'subject_18': 1
}

def compute_ita(lab):
    """Compute Individual Typology Angle (for reference only)"""
    L, a, b = lab[0, 0]
    if b == 0:
        return 0
    return atan((L - 50) / b) * 180 / pi

def find_closest_mst(lab_color):
    """Find closest MST using Delta E (Euclidean distance in LAB space)"""
    L, a, b = lab_color[0, 0]
    
    distances = []
    for ref_lab in mst_lab_palette:
        L_ref, a_ref, b_ref = ref_lab[0, 0]
        # Delta E 1976 formula (Euclidean distance in LAB space)
        delta_e = np.sqrt((L - L_ref)**2 + (a - a_ref)**2 + (b - b_ref)**2)
        distances.append(delta_e)
    
    return distances.index(min(distances)) + 1

def extract_face_skin_color(image_path, model):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    results = model([img_rgb], conf=0.001, imgsz=1280, verbose=False)
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        print(f"No face detected in {image_path}")
        return None

    x1, y1, x2, y2 = map(int, boxes[0].xyxy[0].cpu().numpy())
    face_crop = img_rgb[y1:y2, x1:x2]
    if face_crop.size == 0:
        print(f"Empty face crop in {image_path}")
        return None

    mp_face = mp.solutions.face_mesh
    with mp_face.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True) as face_mesh:
        result = face_mesh.process(face_crop)
        if not result.multi_face_landmarks:
            print(f"No landmarks found in {image_path}")
            return None

        h, w, _ = face_crop.shape
        points = result.multi_face_landmarks[0].landmark
        
        # Improved landmark selection: forehead and cheeks (avoiding features)
        # idxs = [
        #     10,   # forehead center
        #     151,  # forehead left
        #     337,  # forehead right
        #     205,  # right cheek
        #     425,  # left cheek
        # ]

        # 110, 50, 205 Right cheek
        # 330, 280, 435 Left cheek     
        # 10, 109, 338 forhead
        idxs = [110, 50, 205, 330, 280, 435, 10, 109, 338]
        pixels = []

        for idx, point in enumerate(points):
            x = min(w - 1, max(0, int(point.x * w)))
            y = min(h - 1, max(0, int(point.y * h)))
            if idx in idxs:
                pixels.append(face_crop[y, x])
                cv2.circle(face_crop, (x, y), 3, (0, 255, 0), -1)  # Green for selected
            else:
                cv2.circle(face_crop, (x, y), 1, (255, 0, 0), -1)  # Blue for others
            cv2.putText(face_crop, str(idx), (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 255), 1)

        # Filter outliers using percentile filtering
        pixels = np.array(pixels)
        lower = np.percentile(pixels, 10, axis=0)
        upper = np.percentile(pixels, 90, axis=0)
        mask = np.all((pixels >= lower) & (pixels <= upper), axis=1)
        filtered_pixels = pixels[mask]
        
        if len(filtered_pixels) == 0:
            filtered_pixels = pixels  # Fallback if all filtered out
        
        mean_rgb = np.mean(filtered_pixels, axis=0) / 255
        lab = rgb2lab([[mean_rgb]])
        ita = compute_ita(lab)
        mst = find_closest_mst(lab)

        subject_id = Path(image_path).parent.name
        gt_mst = actual_annotations.get(subject_id)

        def draw_color_box(img, top_left, color_rgb, label):
            color_rgb_255 = tuple(int(c * 255) for c in color_rgb)
            x, y = top_left
            cv2.rectangle(img, (x, y), (x + 50, y + 50), color_rgb_255, -1)
            cv2.rectangle(img, (x, y), (x + 50, y + 50), (0, 0, 0), 2)
            cv2.putText(img, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        draw_color_box(face_crop, (0, 0), mst_rgb_palette[mst - 1], f"Pred: {mst}")
        if gt_mst:
            draw_color_box(face_crop, (60, 0), mst_rgb_palette[gt_mst - 1], f"GT: {gt_mst}")

        debug_path = image_path.replace(".jpg", "_landmarks.jpg").replace("subject_", "landmark_subject_")
        os.makedirs(os.path.dirname(debug_path), exist_ok=True)
        cv2.imwrite(debug_path, cv2.cvtColor(face_crop, cv2.COLOR_RGB2BGR))

        return {
            "r": int(mean_rgb[0] * 255),
            "g": int(mean_rgb[1] * 255),
            "b": int(mean_rgb[2] * 255),
            "l": lab[0, 0, 0],
            "a": lab[0, 0, 1],
            "b_lab": lab[0, 0, 2],
            "ita": ita,
            "mst": mst
        }

def batch_process_images(img_dir, output_csv, model_path="yolov12l-face.pt"):
    model = YOLO(model_path)
    model.fuse()
    records = []

    for path in Path(img_dir).rglob("*.jpg"):
        result = extract_face_skin_color(str(path), model)
        if result:
            result["id"] = path.name
            records.append(result)

    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    print(f"Saved skin tone data for {len(df)} faces to {output_csv}")

# Example usage:
for i in range(19):
    subject = f'subject_{i}'
    print(f"Processing {subject}")
    batch_process_images(rf"MonkSkinToneDataset\mst-e_data\{subject}", rf"MonkSkinToneDataset\{subject}.csv")



#  -------------------------------------------------------------------------------------------------------------------

import glob
per_subject_preds = {}
for csv_path in glob.glob("MonkSkinToneDataset/subject_*.csv"):
    # Load and evaluate predictions
    correct = 0
    total = 0
    subject_id = Path(csv_path).stem
    df = pd.read_csv(csv_path)

    if len(df) == 0:
        print(f"[⚠️] No prediction for {subject_id}")
        continue

    # Use majority vote if multiple images per subject
    pred_label = int(df["mst"].mode().iloc[0])
    true_label = actual_annotations.get(subject_id, None)

    if true_label is not None:
        is_correct = (pred_label == true_label)
        per_subject_preds[subject_id] = (pred_label, true_label, is_correct)
        correct += int(is_correct)
        total += 1

# Accuracy summary
print(f"\n✅ Accuracy: {correct}/{total} = {correct / total:.2%}")

# Optional: Show subject-wise results
for subject, (pred, actual, match) in per_subject_preds.items():
    status = "✓" if match else "✗"
    print(f"{subject}: predicted {pred}, actual {actual} → {status}")

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Processing subject_0
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs
No landmarks found in MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_134127250.jpg
Saved skin tone data for 37 faces to MonkSkinToneDataset\subject_0.csv
Processing subject_1
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs
No landmarks found in MonkSkinToneDataset\mst-e_data\subject_1\PXL_20220922_173933735.jpg
No landmarks found in MonkSkinToneDataset\mst-e_data\subject_1\PXL_20220922_173935469.jpg
No landmarks found in MonkSkinToneDataset\mst-e_data\subject_1\PXL_20220922_173937104.jpg
No landmarks found in MonkSkinToneDataset\mst-e_data\subject_1\PXL_20220922_173938355.jpg
No landmarks found in MonkSkinToneDataset\mst-e_data\subject_1\PXL_20220922_173939527.jpg
No landmarks found in MonkSkinToneDataset\mst-e_data\subject_1\PXL_20220922_174126716.jpg
No landmarks found in MonkSkinToneDataset\mst-e_data\subject_1\PXL_20220922_174149039

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from math import atan, pi
from skimage.color import rgb2lab
import mediapipe as mp
from ultralytics import YOLO

# Monk Skin Tone (MST) reference hex codes (1 to 10)
mst_palette = [
    "#f6ede4", "#f3e7db", "#f7ead0", "#eadaba", "#d7bd96",
    "#a07e56", "#825c43", "#604134", "#3a312a", "#292420"
]
mst_rgb_palette = [tuple(int(h[i:i+2], 16)/255 for i in (1, 3, 5)) for h in mst_palette]
mst_lab_palette = [rgb2lab(np.array([[rgb]])) for rgb in mst_rgb_palette]

actual_annotations = {
    'subject_0': 3, 'subject_1': 2, 'subject_2': 8, 'subject_3': 6,
    'subject_4': 9, 'subject_5': 7, 'subject_6': 5, 'subject_7': 4,
    'subject_8': 2, 'subject_9': 4, 'subject_10': 9, 'subject_11': 5,
    'subject_12': 10, 'subject_13': 2, 'subject_14': 6, 'subject_15': 3,
    'subject_16': 1, 'subject_17': 8, 'subject_18': 1
}

def compute_ita(lab):
    """Compute Individual Typology Angle (for reference only)"""
    L, a, b = lab[0, 0]
    if b == 0:
        return 0
    return atan((L - 50) / b) * 180 / pi

def find_closest_mst(lab_color):
    """Find closest MST using Delta E (Euclidean distance in LAB space)"""
    L, a, b = lab_color[0, 0]
    
    distances = []
    for ref_lab in mst_lab_palette:
        L_ref, a_ref, b_ref = ref_lab[0, 0]
        # Delta E 1976 formula (Euclidean distance in LAB space)
        delta_e = np.sqrt((L - L_ref)**2 + (a - a_ref)**2 + (b - b_ref)**2)
        distances.append(delta_e)
    
    return distances.index(min(distances)) + 1

def extract_face_skin_color(image_path, model):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    results = model([img_rgb], conf=0.001, imgsz=1280, verbose=False)
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        print(f"No face detected in {image_path}")
        return None

    x1, y1, x2, y2 = map(int, boxes[0].xyxy[0].cpu().numpy())
    face_crop = img_rgb[y1:y2, x1:x2]
    if face_crop.size == 0:
        print(f"Empty face crop in {image_path}")
        return None

    mp_face = mp.solutions.face_mesh
    with mp_face.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True) as face_mesh:
        result = face_mesh.process(face_crop)
        if not result.multi_face_landmarks:
            print(f"No landmarks found in {image_path}")
            return None

        h, w, _ = face_crop.shape
        points = result.multi_face_landmarks[0].landmark
        
        # Use more reliable landmarks: cheeks and forehead
        # These are verified MediaPipe face mesh indices
        idxs = [
            10,    # upper lip area (good skin reference)
            234,   # left cheek lower
            454,   # right cheek lower  
            109,   # right temple/forehead side
            338,   # left temple/forehead side
            151,   # forehead area
            6,     # forehead center top
        ]
        pixels = []
        sampled_points = []

        # CRITICAL: Sample pixels FIRST, then draw visualizations
        for idx, point in enumerate(points):
            x = min(w - 1, max(0, int(point.x * w)))
            y = min(h - 1, max(0, int(point.y * h)))
            if idx in idxs:
                # Sample 3x3 region around landmark for more robust color
                # DO THIS BEFORE DRAWING!
                for dy in [-1, 0, 1]:
                    for dx in [-1, 0, 1]:
                        px = min(w - 1, max(0, x + dx))
                        py = min(h - 1, max(0, y + dy))
                        pixels.append(face_crop[py, px].copy())  # Copy to avoid reference issues
                sampled_points.append((x, y, idx))
        
        # NOW draw the visualizations AFTER sampling
        for idx, point in enumerate(points):
            x = min(w - 1, max(0, int(point.x * w)))
            y = min(h - 1, max(0, int(point.y * h)))
            if idx in idxs:
                cv2.circle(face_crop, (x, y), 4, (0, 255, 0), -1)  # Green for selected
            else:
                cv2.circle(face_crop, (x, y), 1, (255, 0, 0), -1)  # Blue for others
            cv2.putText(face_crop, str(idx), (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 255), 1)

        # Filter outliers using percentile filtering (less aggressive)
        pixels = np.array(pixels)
        
        # Calculate per-channel percentiles
        lower = np.percentile(pixels, 20, axis=0)
        upper = np.percentile(pixels, 80, axis=0)
        mask = np.all((pixels >= lower) & (pixels <= upper), axis=1)
        filtered_pixels = pixels[mask]
        
        if len(filtered_pixels) < 5:  # Need at least 5 pixels
            filtered_pixels = pixels  # Fallback if too few
        
        mean_rgb = np.mean(filtered_pixels, axis=0) / 255
        
        # Debug: print sampled colors
        print(f"\nImage: {image_path}")
        print(f"Sampled {len(filtered_pixels)} pixels from {len(sampled_points)} landmarks")
        print(f"Mean RGB: {mean_rgb * 255}")
        lab = rgb2lab([[mean_rgb]])
        ita = compute_ita(lab)
        mst = find_closest_mst(lab)

        subject_id = Path(image_path).parent.name
        gt_mst = actual_annotations.get(subject_id)

        def draw_color_box(img, top_left, color_rgb, label):
            color_rgb_255 = tuple(int(c * 255) for c in color_rgb)
            x, y = top_left
            cv2.rectangle(img, (x, y), (x + 50, y + 50), color_rgb_255, -1)
            cv2.rectangle(img, (x, y), (x + 50, y + 50), (0, 0, 0), 2)
            cv2.putText(img, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        draw_color_box(face_crop, (0, 0), mst_rgb_palette[mst - 1], f"Pred: {mst}")
        if gt_mst:
            draw_color_box(face_crop, (60, 0), mst_rgb_palette[gt_mst - 1], f"GT: {gt_mst}")

        debug_path = image_path.replace(".jpg", "_landmarks.jpg").replace("subject_", "landmark_subject_")
        os.makedirs(os.path.dirname(debug_path), exist_ok=True)
        cv2.imwrite(debug_path, cv2.cvtColor(face_crop, cv2.COLOR_RGB2BGR))

        return {
            "r": int(mean_rgb[0] * 255),
            "g": int(mean_rgb[1] * 255),
            "b": int(mean_rgb[2] * 255),
            "l": lab[0, 0, 0],
            "a": lab[0, 0, 1],
            "b_lab": lab[0, 0, 2],
            "ita": ita,
            "mst": mst
        }

def batch_process_images(img_dir, output_csv, model_path="yolov12l-face.pt"):
    model = YOLO(model_path)
    model.fuse()
    records = []

    for path in Path(img_dir).rglob("*.jpg"):
        result = extract_face_skin_color(str(path), model)
        if result:
            result["id"] = path.name
            records.append(result)

    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    print(f"Saved skin tone data for {len(df)} faces to {output_csv}")

# Example usage:
for i in range(19):
    subject = f'subject_{i}'
    print(f"Processing {subject}")
    batch_process_images(rf"MonkSkinToneDataset\mst-e_data\{subject}", rf"MonkSkinToneDataset\{subject}.csv")

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Processing subject_0
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132845778.PORTRAIT.jpg
Sampled 31 pixels from 7 landmarks
Mean RGB: [     229.52      176.65      155.77]

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132858876.PORTRAIT.jpg
Sampled 35 pixels from 7 landmarks
Mean RGB: [     176.94      98.914      84.286]

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132903030.PORTRAIT.jpg
Sampled 34 pixels from 7 landmarks
Mean RGB: [     225.91      201.76      192.53]

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132911049.PORTRAIT.jpg
Sampled 35 pixels from 7 landmarks
Mean RGB: [        219      158.89      140.17]

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132914899.PORTRAIT.jpg
Sampled 36 pixels from 7 landmarks
Mean RGB: [     185.44      110.89      97.917]

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from math import atan, pi
from skimage.color import rgb2lab
import mediapipe as mp
from ultralytics import YOLO

# Monk Skin Tone (MST) reference hex codes (1 to 10)
mst_palette = [
    "#f6ede4", "#f3e7db", "#f7ead0", "#eadaba", "#d7bd96",
    "#a07e56", "#825c43", "#604134", "#3a312a", "#292420"
]
mst_rgb_palette = [tuple(int(h[i:i+2], 16)/255 for i in (1, 3, 5)) for h in mst_palette]
mst_lab_palette = [rgb2lab(np.array([[rgb]])) for rgb in mst_rgb_palette]

actual_annotations = {
    'subject_0': 3, 'subject_1': 2, 'subject_2': 8, 'subject_3': 6,
    'subject_4': 9, 'subject_5': 7, 'subject_6': 5, 'subject_7': 4,
    'subject_8': 2, 'subject_9': 4, 'subject_10': 9, 'subject_11': 5,
    'subject_12': 10, 'subject_13': 2, 'subject_14': 6, 'subject_15': 3,
    'subject_16': 1, 'subject_17': 8, 'subject_18': 1
}

def compute_ita(lab):
    """Compute Individual Typology Angle (for reference only)"""
    L, a, b = lab[0, 0]
    if b == 0:
        return 0
    return atan((L - 50) / b) * 180 / pi

def is_well_lit(rgb_pixel):
    """Check if pixel is well-lit (not too dark, not overexposed)"""
    brightness = np.mean(rgb_pixel)
    return 40 < brightness < 220  # Avoid shadows and highlights

def normalize_illumination(rgb_array):
    """Normalize RGB values to reduce lighting effects using gray world assumption"""
    mean_r, mean_g, mean_b = np.mean(rgb_array, axis=0)
    gray = (mean_r + mean_g + mean_b) / 3
    
    if mean_r > 0 and mean_g > 0 and mean_b > 0:
        scale_r = gray / mean_r
        scale_g = gray / mean_g
        scale_b = gray / mean_b
        
        normalized = rgb_array.copy()
        normalized[:, 0] = np.clip(rgb_array[:, 0] * scale_r, 0, 255)
        normalized[:, 1] = np.clip(rgb_array[:, 1] * scale_g, 0, 255)
        normalized[:, 2] = np.clip(rgb_array[:, 2] * scale_b, 0, 255)
        return normalized
    return rgb_array

def find_closest_mst(lab_color):
    """Find closest MST using Delta E (Euclidean distance in LAB space)"""
    L, a, b = lab_color[0, 0]
    
    distances = []
    for ref_lab in mst_lab_palette:
        L_ref, a_ref, b_ref = ref_lab[0, 0]
        # Delta E 1976 formula (Euclidean distance in LAB space)
        delta_e = np.sqrt((L - L_ref)**2 + (a - a_ref)**2 + (b - b_ref)**2)
        distances.append(delta_e)
    
    return distances.index(min(distances)) + 1

def extract_face_skin_color(image_path, model):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    results = model([img_rgb], conf=0.001, imgsz=1280, verbose=False)
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        print(f"No face detected in {image_path}")
        return None

    x1, y1, x2, y2 = map(int, boxes[0].xyxy[0].cpu().numpy())
    face_crop = img_rgb[y1:y2, x1:x2]
    if face_crop.size == 0:
        print(f"Empty face crop in {image_path}")
        return None

    mp_face = mp.solutions.face_mesh
    with mp_face.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True) as face_mesh:
        result = face_mesh.process(face_crop)
        if not result.multi_face_landmarks:
            print(f"No landmarks found in {image_path}")
            return None

        h, w, _ = face_crop.shape
        points = result.multi_face_landmarks[0].landmark
        
        # Select landmarks on flat, well-lit areas (avoiding nose, eye sockets, shadows)
        # Focus on cheeks and forehead
        idxs = [
            234,   # left cheek (mid)
            454,   # right cheek (mid)
            10,    # forehead center-top
            151,   # cheek/temple area left
            377,   # cheek/temple area right
            93,    # left cheek upper
            323,   # right cheek upper
        ]
        pixels = []
        sampled_points = []
        pixel_positions = []

        # CRITICAL: Sample pixels FIRST, then draw visualizations
        for idx, point in enumerate(points):
            x = min(w - 1, max(0, int(point.x * w)))
            y = min(h - 1, max(0, int(point.y * h)))
            if idx in idxs:
                # Sample 5x5 region around landmark for more robust color
                for dy in range(-2, 3):
                    for dx in range(-2, 3):
                        px = min(w - 1, max(0, x + dx))
                        py = min(h - 1, max(0, y + dy))
                        pixel = face_crop[py, px].copy()
                        
                        # Only include well-lit pixels
                        if is_well_lit(pixel):
                            pixels.append(pixel)
                            pixel_positions.append((px, py))
                
                sampled_points.append((x, y, idx))
        
        if len(pixels) < 10:
            print(f"Not enough well-lit pixels in {image_path}")
            return None
        
        pixels = np.array(pixels)
        
        # Apply illumination normalization
        normalized_pixels = normalize_illumination(pixels)
        
        # Filter outliers using IQR method (more robust than percentile)
        q1 = np.percentile(normalized_pixels, 25, axis=0)
        q3 = np.percentile(normalized_pixels, 75, axis=0)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        mask = np.all((normalized_pixels >= lower_bound) & (normalized_pixels <= upper_bound), axis=1)
        filtered_pixels = normalized_pixels[mask]
        
        if len(filtered_pixels) < 5:
            filtered_pixels = normalized_pixels
        
        mean_rgb = np.mean(filtered_pixels, axis=0) / 255
        
        # Convert to LAB for color matching
        lab = rgb2lab([[mean_rgb]])
        ita = compute_ita(lab)
        mst = find_closest_mst(lab)

        subject_id = Path(image_path).parent.name
        gt_mst = actual_annotations.get(subject_id)

        # Debug: print sampled colors
        print(f"\nImage: {image_path}")
        print(f"Sampled {len(filtered_pixels)} pixels from {len(sampled_points)} landmarks")
        print(f"Mean RGB: {mean_rgb * 255}")
        print(f"LAB: L={lab[0,0,0]:.1f}, a={lab[0,0,1]:.1f}, b={lab[0,0,2]:.1f}")
        print(f"Predicted MST: {mst}, Ground Truth: {gt_mst}")

        # NOW draw the visualizations AFTER sampling
        for idx, point in enumerate(points):
            x = min(w - 1, max(0, int(point.x * w)))
            y = min(h - 1, max(0, int(point.y * h)))
            if idx in idxs:
                cv2.circle(face_crop, (x, y), 5, (0, 255, 0), -1)  # Green for selected
            else:
                cv2.circle(face_crop, (x, y), 1, (100, 100, 255), -1)  # Light blue for others
            cv2.putText(face_crop, str(idx), (x, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 255), 1)
        
        # Mark the well-lit pixels used in computation
        for px, py in pixel_positions[:100]:  # Show first 100 to avoid clutter
            if mask[pixel_positions.index((px, py))] if (px, py) in pixel_positions[:len(mask)] else False:
                cv2.circle(face_crop, (px, py), 1, (255, 255, 0), -1)  # Yellow for used pixels

        def draw_color_box(img, top_left, color_rgb, label):
            color_rgb_255 = tuple(int(c * 255) for c in color_rgb)
            x, y = top_left
            cv2.rectangle(img, (x, y), (x + 50, y + 50), color_rgb_255, -1)
            cv2.rectangle(img, (x, y), (x + 50, y + 50), (0, 0, 0), 2)
            cv2.putText(img, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

        draw_color_box(face_crop, (0, 0), mst_rgb_palette[mst - 1], f"Pred: {mst}")
        if gt_mst:
            draw_color_box(face_crop, (60, 0), mst_rgb_palette[gt_mst - 1], f"GT: {gt_mst}")

        debug_path = image_path.replace(".jpg", "_landmarks.jpg").replace("subject_", "landmark_subject_")
        os.makedirs(os.path.dirname(debug_path), exist_ok=True)
        cv2.imwrite(debug_path, cv2.cvtColor(face_crop, cv2.COLOR_RGB2BGR))

        return {
            "r": int(mean_rgb[0] * 255),
            "g": int(mean_rgb[1] * 255),
            "b": int(mean_rgb[2] * 255),
            "l": lab[0, 0, 0],
            "a": lab[0, 0, 1],
            "b_lab": lab[0, 0, 2],
            "ita": ita,
            "mst": mst,
            "gt_mst": gt_mst if gt_mst else -1,
            "num_pixels": len(filtered_pixels)
        }

def batch_process_images(img_dir, output_csv, model_path="yolov12l-face.pt"):
    model = YOLO(model_path)
    model.fuse()
    records = []

    for path in Path(img_dir).rglob("*.jpg"):
        result = extract_face_skin_color(str(path), model)
        if result:
            result["id"] = path.name
            records.append(result)

    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    
    # Calculate accuracy if ground truth available
    if 'gt_mst' in df.columns:
        valid_preds = df[df['gt_mst'] != -1]
        if len(valid_preds) > 0:
            accuracy = (valid_preds['mst'] == valid_preds['gt_mst']).mean()
            mae = (valid_preds['mst'] - valid_preds['gt_mst']).abs().mean()
            print(f"\nAccuracy: {accuracy:.2%}")
            print(f"Mean Absolute Error: {mae:.2f} MST levels")
    
    print(f"Saved skin tone data for {len(df)} faces to {output_csv}")

# Example usage:
for i in range(19):
    subject = f'subject_{i}'
    print(f"\n{'='*60}")
    print(f"Processing {subject}")
    print(f"{'='*60}")
    batch_process_images(rf"MonkSkinToneDataset\mst-e_data\{subject}", rf"MonkSkinToneDataset\{subject}.csv")

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'


Processing subject_0
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132845778.PORTRAIT.jpg
Sampled 167 pixels from 7 landmarks
Mean RGB: [     143.74      143.68      143.73]
LAB: L=59.7, a=0.0, b=-0.0
Predicted MST: 6, Ground Truth: 3

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132858876.PORTRAIT.jpg
Sampled 125 pixels from 7 landmarks
Mean RGB: [     97.096      82.976      81.272]
LAB: L=36.6, a=5.5, b=3.2
Predicted MST: 8, Ground Truth: 3

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132903030.PORTRAIT.jpg
Sampled 95 pixels from 7 landmarks
Mean RGB: [     84.253      79.389      79.189]
LAB: L=34.2, a=2.0, b=0.8
Predicted MST: 9, Ground Truth: 3

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132911049.PORTRAIT.jpg
Sampled 125 pixels from 7 landmarks
Mean RGB: [     141.46      141.53      141.49]
LAB: L=58.8, a=-0.0, b=0.0
Predicted

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from math import atan, pi
from skimage.color import rgb2lab
import mediapipe as mp
from ultralytics import YOLO

# Monk Skin Tone (MST) reference hex codes (1 to 10)
mst_palette = [
    "#f6ede4", "#f3e7db", "#f7ead0", "#eadaba", "#d7bd96",
    "#a07e56", "#825c43", "#604134", "#3a312a", "#292420"
]
mst_rgb_palette = [tuple(int(h[i:i+2], 16)/255 for i in (1, 3, 5)) for h in mst_palette]
mst_lab_palette = [rgb2lab(np.array([[rgb]])) for rgb in mst_rgb_palette]

actual_annotations = {
    'subject_0': 3, 'subject_1': 2, 'subject_2': 8, 'subject_3': 6,
    'subject_4': 9, 'subject_5': 7, 'subject_6': 5, 'subject_7': 4,
    'subject_8': 2, 'subject_9': 4, 'subject_10': 9, 'subject_11': 5,
    'subject_12': 10, 'subject_13': 2, 'subject_14': 6, 'subject_15': 3,
    'subject_16': 1, 'subject_17': 8, 'subject_18': 1
}

def compute_ita(lab):
    """Compute Individual Typology Angle (for reference only)"""
    L, a, b = lab[0, 0]
    if b == 0:
        return 0
    return atan((L - 50) / b) * 180 / pi

def is_well_lit(rgb_pixel):
    """Check if pixel is well-lit (not too dark, not overexposed, not highly saturated/shadowed)"""
    brightness = np.mean(rgb_pixel)
    # More lenient range to keep more pixels
    if not (30 < brightness < 230):
        return False
    
    # Check for extreme color cast (shadow areas often have blue/green cast)
    std = np.std(rgb_pixel)
    if std > 30:  # Too much color variation suggests non-skin area
        return False
    
    return True

def normalize_illumination(rgb_array):
    """Normalize using chromaticity (color ratios) to preserve hue while reducing lighting effects"""
    # Convert to float
    rgb_float = rgb_array.astype(float)
    
    # Calculate intensity (grayscale)
    intensity = np.mean(rgb_float, axis=1, keepdims=True)
    
    # Avoid division by zero
    intensity = np.where(intensity > 0, intensity, 1)
    
    # Calculate chromaticity (rgb / intensity)
    # This preserves color ratios while normalizing brightness
    chromaticity = rgb_float / intensity
    
    # Reconstruct with a standard mid-tone brightness (around 120)
    # This puts all images on similar brightness scale while preserving color
    target_brightness = 120
    normalized = chromaticity * target_brightness
    
    return np.clip(normalized, 0, 255)

def find_closest_mst(lab_color):
    """Find closest MST using Delta E (Euclidean distance in LAB space)"""
    L, a, b = lab_color[0, 0]
    
    distances = []
    for ref_lab in mst_lab_palette:
        L_ref, a_ref, b_ref = ref_lab[0, 0]
        # Delta E 1976 formula (Euclidean distance in LAB space)
        delta_e = np.sqrt((L - L_ref)**2 + (a - a_ref)**2 + (b - b_ref)**2)
        distances.append(delta_e)
    
    return distances.index(min(distances)) + 1

def extract_face_skin_color(image_path, model):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    results = model([img_rgb], conf=0.001, imgsz=1280, verbose=False)
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        print(f"No face detected in {image_path}")
        return None

    x1, y1, x2, y2 = map(int, boxes[0].xyxy[0].cpu().numpy())
    face_crop = img_rgb[y1:y2, x1:x2]
    if face_crop.size == 0:
        print(f"Empty face crop in {image_path}")
        return None

    mp_face = mp.solutions.face_mesh
    with mp_face.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True) as face_mesh:
        result = face_mesh.process(face_crop)
        if not result.multi_face_landmarks:
            print(f"No landmarks found in {image_path}")
            return None

        h, w, _ = face_crop.shape
        points = result.multi_face_landmarks[0].landmark
        
        # Select landmarks on flat, well-lit areas (avoiding nose, eye sockets, shadows)
        # Focus on cheeks and forehead
        idxs = [
            234,   # left cheek (mid)
            454,   # right cheek (mid)
            10,    # forehead center-top
            151,   # cheek/temple area left
            377,   # cheek/temple area right
            93,    # left cheek upper
            323,   # right cheek upper
        ]
        pixels = []
        sampled_points = []
        pixel_positions = []

        # CRITICAL: Sample pixels FIRST, then draw visualizations
        for idx, point in enumerate(points):
            x = min(w - 1, max(0, int(point.x * w)))
            y = min(h - 1, max(0, int(point.y * h)))
            if idx in idxs:
                # Sample 5x5 region around landmark for more robust color
                for dy in range(-2, 3):
                    for dx in range(-2, 3):
                        px = min(w - 1, max(0, x + dx))
                        py = min(h - 1, max(0, y + dy))
                        pixel = face_crop[py, px].copy()
                        
                        # Only include well-lit pixels
                        if is_well_lit(pixel):
                            pixels.append(pixel)
                            pixel_positions.append((px, py))
                
                sampled_points.append((x, y, idx))
        
        if len(pixels) < 10:
            print(f"Not enough well-lit pixels in {image_path}")
            return None
        
        pixels = np.array(pixels)
        
        # Apply illumination normalization
        normalized_pixels = normalize_illumination(pixels)
        
        # Filter outliers using IQR method (more robust than percentile)
        q1 = np.percentile(normalized_pixels, 25, axis=0)
        q3 = np.percentile(normalized_pixels, 75, axis=0)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        mask = np.all((normalized_pixels >= lower_bound) & (normalized_pixels <= upper_bound), axis=1)
        filtered_pixels = normalized_pixels[mask]
        
        if len(filtered_pixels) < 5:
            filtered_pixels = normalized_pixels
        
        mean_rgb = np.mean(filtered_pixels, axis=0) / 255
        
        # Convert to LAB for color matching
        lab = rgb2lab([[mean_rgb]])
        ita = compute_ita(lab)
        mst = find_closest_mst(lab)

        subject_id = Path(image_path).parent.name
        gt_mst = actual_annotations.get(subject_id)

        # Debug: print sampled colors
        print(f"\nImage: {image_path}")
        print(f"Sampled {len(filtered_pixels)} pixels from {len(sampled_points)} landmarks")
        print(f"Mean RGB: {mean_rgb * 255}")
        print(f"LAB: L={lab[0,0,0]:.1f}, a={lab[0,0,1]:.1f}, b={lab[0,0,2]:.1f}")
        print(f"Predicted MST: {mst}, Ground Truth: {gt_mst}")

        # NOW draw the visualizations AFTER sampling
        for idx, point in enumerate(points):
            x = min(w - 1, max(0, int(point.x * w)))
            y = min(h - 1, max(0, int(point.y * h)))
            if idx in idxs:
                cv2.circle(face_crop, (x, y), 5, (0, 255, 0), -1)  # Green for selected
            else:
                cv2.circle(face_crop, (x, y), 1, (100, 100, 255), -1)  # Light blue for others
            cv2.putText(face_crop, str(idx), (x, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 255), 1)
        
        # Mark the well-lit pixels used in computation
        for px, py in pixel_positions[:100]:  # Show first 100 to avoid clutter
            if mask[pixel_positions.index((px, py))] if (px, py) in pixel_positions[:len(mask)] else False:
                cv2.circle(face_crop, (px, py), 1, (255, 255, 0), -1)  # Yellow for used pixels

        def draw_color_box(img, top_left, color_rgb, label):
            color_rgb_255 = tuple(int(c * 255) for c in color_rgb)
            x, y = top_left
            cv2.rectangle(img, (x, y), (x + 50, y + 50), color_rgb_255, -1)
            cv2.rectangle(img, (x, y), (x + 50, y + 50), (0, 0, 0), 2)
            cv2.putText(img, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

        draw_color_box(face_crop, (0, 0), mst_rgb_palette[mst - 1], f"Pred: {mst}")
        if gt_mst:
            draw_color_box(face_crop, (60, 0), mst_rgb_palette[gt_mst - 1], f"GT: {gt_mst}")

        debug_path = image_path.replace(".jpg", "_landmarks.jpg").replace("subject_", "landmark_subject_")
        os.makedirs(os.path.dirname(debug_path), exist_ok=True)
        cv2.imwrite(debug_path, cv2.cvtColor(face_crop, cv2.COLOR_RGB2BGR))

        return {
            "r": int(mean_rgb[0] * 255),
            "g": int(mean_rgb[1] * 255),
            "b": int(mean_rgb[2] * 255),
            "l": lab[0, 0, 0],
            "a": lab[0, 0, 1],
            "b_lab": lab[0, 0, 2],
            "ita": ita,
            "mst": mst,
            "gt_mst": gt_mst if gt_mst else -1,
            "num_pixels": len(filtered_pixels)
        }

def batch_process_images(img_dir, output_csv, model_path="yolov12l-face.pt"):
    model = YOLO(model_path)
    model.fuse()
    records = []

    for path in Path(img_dir).rglob("*.jpg"):
        result = extract_face_skin_color(str(path), model)
        if result:
            result["id"] = path.name
            records.append(result)

    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    
    # Calculate accuracy if ground truth available
    if 'gt_mst' in df.columns:
        valid_preds = df[df['gt_mst'] != -1]
        if len(valid_preds) > 0:
            accuracy = (valid_preds['mst'] == valid_preds['gt_mst']).mean()
            mae = (valid_preds['mst'] - valid_preds['gt_mst']).abs().mean()
            print(f"\nAccuracy: {accuracy:.2%}")
            print(f"Mean Absolute Error: {mae:.2f} MST levels")
    
    print(f"Saved skin tone data for {len(df)} faces to {output_csv}")

# Example usage:
for i in range(19):
    subject = f'subject_{i}'
    print(f"\n{'='*60}")
    print(f"Processing {subject}")
    print(f"{'='*60}")
    batch_process_images(rf"MonkSkinToneDataset\mst-e_data\{subject}", rf"MonkSkinToneDataset\{subject}.csv")

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'


Processing subject_0
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132845778.PORTRAIT.jpg
Sampled 131 pixels from 7 landmarks
Mean RGB: [     157.39      107.11      95.499]
LAB: L=50.2, a=18.6, b=14.6
Predicted MST: 7, Ground Truth: 3

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132858876.PORTRAIT.jpg
Sampled 59 pixels from 7 landmarks
Mean RGB: [      163.4      104.55      92.043]
LAB: L=50.2, a=22.1, b=16.8
Predicted MST: 7, Ground Truth: 3

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132903030.PORTRAIT.jpg
Sampled 66 pixels from 7 landmarks
Mean RGB: [     197.99      86.367      75.489]
LAB: L=51.0, a=43.8, b=28.4
Predicted MST: 7, Ground Truth: 3

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132911049.PORTRAIT.jpg
Sampled 15 pixels from 7 landmarks
Mean RGB: [      131.4      120.04      108.56]
LAB: L=51.1, a=2.3, b=7.9
Predict

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from math import atan, pi
from skimage.color import rgb2lab
import mediapipe as mp
from ultralytics import YOLO

# Monk Skin Tone (MST) reference hex codes (1 to 10)
mst_palette = [
    "#f6ede4", "#f3e7db", "#f7ead0", "#eadaba", "#d7bd96",
    "#a07e56", "#825c43", "#604134", "#3a312a", "#292420"
]
mst_rgb_palette = [tuple(int(h[i:i+2], 16)/255 for i in (1, 3, 5)) for h in mst_palette]
mst_lab_palette = [rgb2lab(np.array([[rgb]])) for rgb in mst_rgb_palette]

actual_annotations = {
    'subject_0': 3, 'subject_1': 2, 'subject_2': 8, 'subject_3': 6,
    'subject_4': 9, 'subject_5': 7, 'subject_6': 5, 'subject_7': 4,
    'subject_8': 2, 'subject_9': 4, 'subject_10': 9, 'subject_11': 5,
    'subject_12': 10, 'subject_13': 2, 'subject_14': 6, 'subject_15': 3,
    'subject_16': 1, 'subject_17': 8, 'subject_18': 1
}

def compute_ita(lab):
    """Compute Individual Typology Angle (for reference only)"""
    L, a, b = lab[0, 0]
    if b == 0:
        return 0
    return atan((L - 50) / b) * 180 / pi

def is_well_lit(rgb_pixel):
    """Check if pixel is well-lit (not too dark, not overexposed, not highly saturated/shadowed)"""
    brightness = np.mean(rgb_pixel)
    # More lenient range to keep more pixels
    if not (30 < brightness < 230):
        return False
    
    # Check for extreme color cast (shadow areas often have blue/green cast)
    std = np.std(rgb_pixel)
    if std > 30:  # Too much color variation suggests non-skin area
        return False
    
    return True

def normalize_illumination(rgb_array):
    """Minimal normalization - just remove extreme color casts"""
    rgb_float = rgb_array.astype(float)
    
    # Calculate per-channel medians
    median_r = np.median(rgb_float[:, 0])
    median_g = np.median(rgb_float[:, 1])
    median_b = np.median(rgb_float[:, 2])
    
    # Detect and correct color casts (e.g., blue-ish shadows, yellow-ish highlights)
    overall_median = (median_r + median_g + median_b) / 3
    
    # Only correct if there's a significant color cast (>10% deviation)
    if abs(median_r - overall_median) > overall_median * 0.1 or \
       abs(median_g - overall_median) > overall_median * 0.1 or \
       abs(median_b - overall_median) > overall_median * 0.1:
        
        # Gentle color balance
        scale_r = overall_median / max(median_r, 1) if median_r > 0 else 1
        scale_g = overall_median / max(median_g, 1) if median_g > 0 else 1
        scale_b = overall_median / max(median_b, 1) if median_b > 0 else 1
        
        # Apply with reduced intensity (0.5 = 50% correction)
        scale_r = 1.0 + (scale_r - 1.0) * 0.5
        scale_g = 1.0 + (scale_g - 1.0) * 0.5
        scale_b = 1.0 + (scale_b - 1.0) * 0.5
        
        rgb_float[:, 0] = rgb_float[:, 0] * scale_r
        rgb_float[:, 1] = rgb_float[:, 1] * scale_g
        rgb_float[:, 2] = rgb_float[:, 2] * scale_b
    
    return np.clip(rgb_float, 0, 255)

def find_closest_mst(lab_color):
    """Find closest MST using weighted Delta E with emphasis on color over brightness"""
    L, a, b = lab_color[0, 0]
    
    distances = []
    for ref_lab in mst_lab_palette:
        L_ref, a_ref, b_ref = ref_lab[0, 0]
        
        # Weighted distance: color (a, b) is 2x more important than lightness (L)
        # This helps distinguish skin tones even under varying lighting
        delta_L = (L - L_ref) * 0.5  # Reduced weight for lightness
        delta_a = (a - a_ref) * 1.0  # Full weight for red-green
        delta_b = (b - b_ref) * 1.0  # Full weight for yellow-blue
        
        delta_e = np.sqrt(delta_L**2 + delta_a**2 + delta_b**2)
        distances.append(delta_e)
    
    return distances.index(min(distances)) + 1

def extract_face_skin_color(image_path, model):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    results = model([img_rgb], conf=0.001, imgsz=1280, verbose=False)
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        print(f"No face detected in {image_path}")
        return None

    x1, y1, x2, y2 = map(int, boxes[0].xyxy[0].cpu().numpy())
    face_crop = img_rgb[y1:y2, x1:x2]
    if face_crop.size == 0:
        print(f"Empty face crop in {image_path}")
        return None

    mp_face = mp.solutions.face_mesh
    with mp_face.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True) as face_mesh:
        result = face_mesh.process(face_crop)
        if not result.multi_face_landmarks:
            print(f"No landmarks found in {image_path}")
            return None

        h, w, _ = face_crop.shape
        points = result.multi_face_landmarks[0].landmark
        
        # Select landmarks on flat, well-lit areas (avoiding nose, eye sockets, shadows)
        # Focus on cheeks and forehead
        idxs = [
            234,   # left cheek (mid)
            454,   # right cheek (mid)
            10,    # forehead center-top
            151,   # cheek/temple area left
            377,   # cheek/temple area right
            93,    # left cheek upper
            323,   # right cheek upper
        ]
        pixels = []
        sampled_points = []
        pixel_positions = []

        # CRITICAL: Sample pixels FIRST, then draw visualizations
        for idx, point in enumerate(points):
            x = min(w - 1, max(0, int(point.x * w)))
            y = min(h - 1, max(0, int(point.y * h)))
            if idx in idxs:
                # Sample 5x5 region around landmark for more robust color
                for dy in range(-2, 3):
                    for dx in range(-2, 3):
                        px = min(w - 1, max(0, x + dx))
                        py = min(h - 1, max(0, y + dy))
                        pixel = face_crop[py, px].copy()
                        
                        # Only include well-lit pixels
                        if is_well_lit(pixel):
                            pixels.append(pixel)
                            pixel_positions.append((px, py))
                
                sampled_points.append((x, y, idx))
        
        if len(pixels) < 10:
            print(f"Not enough well-lit pixels in {image_path}")
            return None
        
        pixels = np.array(pixels)
        
        # Apply illumination normalization
        normalized_pixels = normalize_illumination(pixels)
        
        # Filter outliers using IQR method (more robust than percentile)
        q1 = np.percentile(normalized_pixels, 25, axis=0)
        q3 = np.percentile(normalized_pixels, 75, axis=0)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        mask = np.all((normalized_pixels >= lower_bound) & (normalized_pixels <= upper_bound), axis=1)
        filtered_pixels = normalized_pixels[mask]
        
        if len(filtered_pixels) < 5:
            filtered_pixels = normalized_pixels
        
        mean_rgb = np.mean(filtered_pixels, axis=0) / 255
        
        # Convert to LAB for color matching
        lab = rgb2lab([[mean_rgb]])
        ita = compute_ita(lab)
        mst = find_closest_mst(lab)

        subject_id = Path(image_path).parent.name
        gt_mst = actual_annotations.get(subject_id)

        # Debug: print sampled colors
        print(f"\nImage: {image_path}")
        print(f"Sampled {len(filtered_pixels)} pixels from {len(sampled_points)} landmarks")
        print(f"Mean RGB: {mean_rgb * 255}")
        print(f"LAB: L={lab[0,0,0]:.1f}, a={lab[0,0,1]:.1f}, b={lab[0,0,2]:.1f}")
        print(f"Predicted MST: {mst}, Ground Truth: {gt_mst}")

        # NOW draw the visualizations AFTER sampling
        for idx, point in enumerate(points):
            x = min(w - 1, max(0, int(point.x * w)))
            y = min(h - 1, max(0, int(point.y * h)))
            if idx in idxs:
                cv2.circle(face_crop, (x, y), 5, (0, 255, 0), -1)  # Green for selected
            else:
                cv2.circle(face_crop, (x, y), 1, (100, 100, 255), -1)  # Light blue for others
            cv2.putText(face_crop, str(idx), (x, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 255), 1)
        
        # Mark the well-lit pixels used in computation
        for px, py in pixel_positions[:100]:  # Show first 100 to avoid clutter
            if mask[pixel_positions.index((px, py))] if (px, py) in pixel_positions[:len(mask)] else False:
                cv2.circle(face_crop, (px, py), 1, (255, 255, 0), -1)  # Yellow for used pixels

        def draw_color_box(img, top_left, color_rgb, label):
            color_rgb_255 = tuple(int(c * 255) for c in color_rgb)
            x, y = top_left
            cv2.rectangle(img, (x, y), (x + 50, y + 50), color_rgb_255, -1)
            cv2.rectangle(img, (x, y), (x + 50, y + 50), (0, 0, 0), 2)
            cv2.putText(img, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

        draw_color_box(face_crop, (0, 0), mst_rgb_palette[mst - 1], f"Pred: {mst}")
        if gt_mst:
            draw_color_box(face_crop, (60, 0), mst_rgb_palette[gt_mst - 1], f"GT: {gt_mst}")

        debug_path = image_path.replace(".jpg", "_landmarks.jpg").replace("subject_", "landmark_subject_")
        os.makedirs(os.path.dirname(debug_path), exist_ok=True)
        cv2.imwrite(debug_path, cv2.cvtColor(face_crop, cv2.COLOR_RGB2BGR))

        return {
            "r": int(mean_rgb[0] * 255),
            "g": int(mean_rgb[1] * 255),
            "b": int(mean_rgb[2] * 255),
            "l": lab[0, 0, 0],
            "a": lab[0, 0, 1],
            "b_lab": lab[0, 0, 2],
            "ita": ita,
            "mst": mst,
            "gt_mst": gt_mst if gt_mst else -1,
            "num_pixels": len(filtered_pixels)
        }

def batch_process_images(img_dir, output_csv, model_path="yolov12l-face.pt"):
    model = YOLO(model_path)
    model.fuse()
    records = []

    for path in Path(img_dir).rglob("*.jpg"):
        result = extract_face_skin_color(str(path), model)
        if result:
            result["id"] = path.name
            records.append(result)

    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    
    # Calculate accuracy if ground truth available
    if 'gt_mst' in df.columns:
        valid_preds = df[df['gt_mst'] != -1]
        if len(valid_preds) > 0:
            accuracy = (valid_preds['mst'] == valid_preds['gt_mst']).mean()
            mae = (valid_preds['mst'] - valid_preds['gt_mst']).abs().mean()
            print(f"\nAccuracy: {accuracy:.2%}")
            print(f"Mean Absolute Error: {mae:.2f} MST levels")
    
    print(f"Saved skin tone data for {len(df)} faces to {output_csv}")

# Example usage:
for i in range(19):
    subject = f'subject_{i}'
    print(f"\n{'='*60}")
    print(f"Processing {subject}")
    print(f"{'='*60}")
    batch_process_images(rf"MonkSkinToneDataset\mst-e_data\{subject}", rf"MonkSkinToneDataset\{subject}.csv")

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'


Processing subject_0
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132845778.PORTRAIT.jpg
Sampled 131 pixels from 7 landmarks
Mean RGB: [     149.57      136.39      127.01]
LAB: L=57.7, a=3.3, b=6.8
Predicted MST: 8, Ground Truth: 3

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132858876.PORTRAIT.jpg
Sampled 59 pixels from 7 landmarks
Mean RGB: [     174.09      143.88      136.42]
LAB: L=62.3, a=10.1, b=8.2
Predicted MST: 7, Ground Truth: 3

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132903030.PORTRAIT.jpg
Sampled 66 pixels from 7 landmarks
Mean RGB: [     85.715      61.604      55.319]
LAB: L=28.5, a=9.6, b=8.0
Predicted MST: 8, Ground Truth: 3

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132911049.PORTRAIT.jpg
Sampled 21 pixels from 7 landmarks
Mean RGB: [     245.14      225.05      204.05]
LAB: L=90.6, a=3.5, b=12.8
Predicted M

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from math import atan, pi
from skimage.color import rgb2lab
import mediapipe as mp
from ultralytics import YOLO

# Monk Skin Tone (MST) reference hex codes (1 to 10)
mst_palette = [
    "#f6ede4", "#f3e7db", "#f7ead0", "#eadaba", "#d7bd96",
    "#a07e56", "#825c43", "#604134", "#3a312a", "#292420"
]
mst_rgb_palette = [tuple(int(h[i:i+2], 16)/255 for i in (1, 3, 5)) for h in mst_palette]
mst_lab_palette = [rgb2lab(np.array([[rgb]])) for rgb in mst_rgb_palette]

actual_annotations = {
    'subject_0': 3, 'subject_1': 2, 'subject_2': 8, 'subject_3': 6,
    'subject_4': 9, 'subject_5': 7, 'subject_6': 5, 'subject_7': 4,
    'subject_8': 2, 'subject_9': 4, 'subject_10': 9, 'subject_11': 5,
    'subject_12': 10, 'subject_13': 2, 'subject_14': 6, 'subject_15': 3,
    'subject_16': 1, 'subject_17': 8, 'subject_18': 1
}

def compute_ita(lab):
    """Compute Individual Typology Angle (for reference only)"""
    L, a, b = lab[0, 0]
    if b == 0:
        return 0
    return atan((L - 50) / b) * 180 / pi

def is_well_lit(rgb_pixel):
    """Check if pixel is well-lit (not too dark, not overexposed, not highly saturated/shadowed)"""
    brightness = np.mean(rgb_pixel)
    # More lenient range to keep more pixels
    if not (30 < brightness < 230):
        return False
    
    # Check for extreme color cast (shadow areas often have blue/green cast)
    std = np.std(rgb_pixel)
    if std > 30:  # Too much color variation suggests non-skin area
        return False
    
    return True

def normalize_illumination(rgb_array):
    """Adaptive illumination normalization with brightness compensation"""
    rgb_float = rgb_array.astype(float)
    
    # Calculate current median brightness
    median_brightness = np.median(np.mean(rgb_float, axis=1))
    
    # Target brightness for good exposure (empirically determined)
    target_brightness = 140
    
    # Calculate scaling factor with adaptive strength
    # More aggressive correction for very dark images, gentle for well-lit images
    if median_brightness < 50:
        # Very dark - strong correction needed
        scale = (target_brightness / max(median_brightness, 1)) ** 0.7
    elif median_brightness < 100:
        # Underexposed - moderate correction
        scale = (target_brightness / max(median_brightness, 1)) ** 0.5
    elif median_brightness > 180:
        # Overexposed - gentle reduction
        scale = (target_brightness / median_brightness) ** 0.3
    else:
        # Well exposed - minimal adjustment
        scale = (target_brightness / median_brightness) ** 0.4
    
    # Apply scaling
    normalized = rgb_float * scale
    
    # Color cast correction (minimal)
    median_r = np.median(normalized[:, 0])
    median_g = np.median(normalized[:, 1])
    median_b = np.median(normalized[:, 2])
    overall_median = (median_r + median_g + median_b) / 3
    
    # Only correct significant color casts
    if overall_median > 10:  # Avoid division by near-zero
        if abs(median_r - overall_median) > overall_median * 0.15:
            scale_r = 1.0 + (overall_median / max(median_r, 1) - 1.0) * 0.3
            normalized[:, 0] *= scale_r
        if abs(median_g - overall_median) > overall_median * 0.15:
            scale_g = 1.0 + (overall_median / max(median_g, 1) - 1.0) * 0.3
            normalized[:, 1] *= scale_g
        if abs(median_b - overall_median) > overall_median * 0.15:
            scale_b = 1.0 + (overall_median / max(median_b, 1) - 1.0) * 0.3
            normalized[:, 2] *= scale_b
    
    return np.clip(normalized, 0, 255)

def find_closest_mst(lab_color):
    """Find closest MST using weighted Delta E with emphasis on color over brightness"""
    L, a, b = lab_color[0, 0]
    
    distances = []
    for ref_lab in mst_lab_palette:
        L_ref, a_ref, b_ref = ref_lab[0, 0]
        
        # Weighted distance: color (a, b) is 2x more important than lightness (L)
        # This helps distinguish skin tones even under varying lighting
        delta_L = (L - L_ref) * 0.5  # Reduced weight for lightness
        delta_a = (a - a_ref) * 1.0  # Full weight for red-green
        delta_b = (b - b_ref) * 1.0  # Full weight for yellow-blue
        
        delta_e = np.sqrt(delta_L**2 + delta_a**2 + delta_b**2)
        distances.append(delta_e)
    
    return distances.index(min(distances)) + 1

def extract_face_skin_color(image_path, model):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    results = model([img_rgb], conf=0.001, imgsz=1280, verbose=False)
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        print(f"No face detected in {image_path}")
        return None

    x1, y1, x2, y2 = map(int, boxes[0].xyxy[0].cpu().numpy())
    face_crop = img_rgb[y1:y2, x1:x2]
    if face_crop.size == 0:
        print(f"Empty face crop in {image_path}")
        return None

    mp_face = mp.solutions.face_mesh
    with mp_face.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True) as face_mesh:
        result = face_mesh.process(face_crop)
        if not result.multi_face_landmarks:
            print(f"No landmarks found in {image_path}")
            return None

        h, w, _ = face_crop.shape
        points = result.multi_face_landmarks[0].landmark
        
        # Select landmarks on flat, well-lit areas (avoiding nose, eye sockets, shadows)
        # Focus on cheeks and forehead
        idxs = [
            234,   # left cheek (mid)
            454,   # right cheek (mid)
            10,    # forehead center-top
            151,   # cheek/temple area left
            377,   # cheek/temple area right
            93,    # left cheek upper
            323,   # right cheek upper
        ]
        pixels = []
        sampled_points = []
        pixel_positions = []

        # CRITICAL: Sample pixels FIRST, then draw visualizations
        for idx, point in enumerate(points):
            x = min(w - 1, max(0, int(point.x * w)))
            y = min(h - 1, max(0, int(point.y * h)))
            if idx in idxs:
                # Sample 5x5 region around landmark for more robust color
                for dy in range(-2, 3):
                    for dx in range(-2, 3):
                        px = min(w - 1, max(0, x + dx))
                        py = min(h - 1, max(0, y + dy))
                        pixel = face_crop[py, px].copy()
                        
                        # Only include well-lit pixels
                        if is_well_lit(pixel):
                            pixels.append(pixel)
                            pixel_positions.append((px, py))
                
                sampled_points.append((x, y, idx))
        
        if len(pixels) < 10:
            print(f"Not enough well-lit pixels in {image_path}")
            return None
        
        pixels = np.array(pixels)
        
        # Apply illumination normalization
        normalized_pixels = normalize_illumination(pixels)
        
        # Filter outliers using IQR method (more robust than percentile)
        q1 = np.percentile(normalized_pixels, 25, axis=0)
        q3 = np.percentile(normalized_pixels, 75, axis=0)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        mask = np.all((normalized_pixels >= lower_bound) & (normalized_pixels <= upper_bound), axis=1)
        filtered_pixels = normalized_pixels[mask]
        
        if len(filtered_pixels) < 5:
            filtered_pixels = normalized_pixels
        
        mean_rgb = np.mean(filtered_pixels, axis=0) / 255
        
        # Convert to LAB for color matching
        lab = rgb2lab([[mean_rgb]])
        ita = compute_ita(lab)
        mst = find_closest_mst(lab)

        subject_id = Path(image_path).parent.name
        gt_mst = actual_annotations.get(subject_id)

        # Debug: print sampled colors
        print(f"\nImage: {image_path}")
        print(f"Sampled {len(filtered_pixels)} pixels from {len(sampled_points)} landmarks")
        print(f"Mean RGB: {mean_rgb * 255}")
        print(f"LAB: L={lab[0,0,0]:.1f}, a={lab[0,0,1]:.1f}, b={lab[0,0,2]:.1f}")
        print(f"Predicted MST: {mst}, Ground Truth: {gt_mst}")

        # NOW draw the visualizations AFTER sampling
        for idx, point in enumerate(points):
            x = min(w - 1, max(0, int(point.x * w)))
            y = min(h - 1, max(0, int(point.y * h)))
            if idx in idxs:
                cv2.circle(face_crop, (x, y), 5, (0, 255, 0), -1)  # Green for selected
            else:
                cv2.circle(face_crop, (x, y), 1, (100, 100, 255), -1)  # Light blue for others
            cv2.putText(face_crop, str(idx), (x, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 255), 1)
        
        # Mark the well-lit pixels used in computation
        for px, py in pixel_positions[:100]:  # Show first 100 to avoid clutter
            if mask[pixel_positions.index((px, py))] if (px, py) in pixel_positions[:len(mask)] else False:
                cv2.circle(face_crop, (px, py), 1, (255, 255, 0), -1)  # Yellow for used pixels

        def draw_color_box(img, top_left, color_rgb, label):
            color_rgb_255 = tuple(int(c * 255) for c in color_rgb)
            x, y = top_left
            cv2.rectangle(img, (x, y), (x + 50, y + 50), color_rgb_255, -1)
            cv2.rectangle(img, (x, y), (x + 50, y + 50), (0, 0, 0), 2)
            cv2.putText(img, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

        draw_color_box(face_crop, (0, 0), mst_rgb_palette[mst - 1], f"Pred: {mst}")
        if gt_mst:
            draw_color_box(face_crop, (60, 0), mst_rgb_palette[gt_mst - 1], f"GT: {gt_mst}")

        debug_path = image_path.replace(".jpg", "_landmarks.jpg").replace("subject_", "landmark_subject_")
        os.makedirs(os.path.dirname(debug_path), exist_ok=True)
        cv2.imwrite(debug_path, cv2.cvtColor(face_crop, cv2.COLOR_RGB2BGR))

        return {
            "r": int(mean_rgb[0] * 255),
            "g": int(mean_rgb[1] * 255),
            "b": int(mean_rgb[2] * 255),
            "l": lab[0, 0, 0],
            "a": lab[0, 0, 1],
            "b_lab": lab[0, 0, 2],
            "ita": ita,
            "mst": mst,
            "gt_mst": gt_mst if gt_mst else -1,
            "num_pixels": len(filtered_pixels)
        }

def batch_process_images(img_dir, output_csv, model_path="yolov12l-face.pt"):
    model = YOLO(model_path)
    model.fuse()
    records = []

    for path in Path(img_dir).rglob("*.jpg"):
        result = extract_face_skin_color(str(path), model)
        if result:
            result["id"] = path.name
            records.append(result)

    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    
    # Calculate accuracy if ground truth available
    if 'gt_mst' in df.columns:
        valid_preds = df[df['gt_mst'] != -1]
        if len(valid_preds) > 0:
            accuracy = (valid_preds['mst'] == valid_preds['gt_mst']).mean()
            mae = (valid_preds['mst'] - valid_preds['gt_mst']).abs().mean()
            print(f"\nAccuracy: {accuracy:.2%}")
            print(f"Mean Absolute Error: {mae:.2f} MST levels")
    
    print(f"Saved skin tone data for {len(df)} faces to {output_csv}")

# Example usage:
for i in range(19):
    subject = f'subject_{i}'
    print(f"\n{'='*60}")
    print(f"Processing {subject}")
    print(f"{'='*60}")
    batch_process_images(rf"MonkSkinToneDataset\mst-e_data\{subject}", rf"MonkSkinToneDataset\{subject}.csv")

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'


Processing subject_0
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132845778.PORTRAIT.jpg
Sampled 131 pixels from 7 landmarks
Mean RGB: [      186.1      150.52      145.45]
LAB: L=65.3, a=12.5, b=7.7
Predicted MST: 2, Ground Truth: 3

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132858876.PORTRAIT.jpg
Sampled 59 pixels from 7 landmarks
Mean RGB: [     163.17      125.37      112.24]
LAB: L=55.8, a=12.7, b=12.8
Predicted MST: 7, Ground Truth: 3

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132903030.PORTRAIT.jpg
Sampled 66 pixels from 7 landmarks
Mean RGB: [     131.99      81.375      72.156]
LAB: L=40.1, a=20.1, b=14.3
Predicted MST: 8, Ground Truth: 3

Image: MonkSkinToneDataset\mst-e_data\subject_0\PXL_20220922_132911049.PORTRAIT.jpg
Sampled 21 pixels from 7 landmarks
Mean RGB: [     212.76      195.32      177.09]
LAB: L=79.9, a=3.1, b=11.4
Predict

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from math import atan, pi
from skimage.color import rgb2lab
import mediapipe as mp
from ultralytics import YOLO

# Monk Skin Tone (MST) reference hex codes (1 to 10)
mst_palette = [
    "#f6ede4", "#f3e7db", "#f7ead0", "#eadaba", "#d7bd96",
    "#a07e56", "#825c43", "#604134", "#3a312a", "#292420"
]
mst_rgb_palette = [tuple(int(h[i:i+2], 16)/255 for i in (1, 3, 5)) for h in mst_palette]
mst_lab_palette = [rgb2lab(np.array([[rgb]])) for rgb in mst_rgb_palette]

actual_annotations = {
    'subject_0': 3, 'subject_1': 2, 'subject_2': 8, 'subject_3': 6,
    'subject_4': 9, 'subject_5': 7, 'subject_6': 5, 'subject_7': 4,
    'subject_8': 2, 'subject_9': 4, 'subject_10': 9, 'subject_11': 5,
    'subject_12': 10, 'subject_13': 2, 'subject_14': 6, 'subject_15': 3,
    'subject_16': 1, 'subject_17': 8, 'subject_18': 1
}

def compute_ita(lab):
    """Compute Individual Typology Angle (for reference only)"""
    L, a, b = lab[0, 0]
    if b == 0:
        return 0
    return atan((L - 50) / b) * 180 / pi

def is_well_lit(rgb_pixel):
    """Check if pixel is well-lit (not too dark, not overexposed, not highly saturated/shadowed)"""
    brightness = np.mean(rgb_pixel)
    # More lenient range to keep more pixels
    if not (30 < brightness < 230):
        return False
    
    # Check for extreme color cast (shadow areas often have blue/green cast)
    std = np.std(rgb_pixel)
    if std > 30:  # Too much color variation suggests non-skin area
        return False
    
    return True

def normalize_illumination(rgb_array):
    """Adaptive illumination normalization with brightness compensation"""
    rgb_float = rgb_array.astype(float)
    
    # Calculate current median brightness
    median_brightness = np.median(np.mean(rgb_float, axis=1))
    
    # Target brightness for good exposure (empirically determined)
    target_brightness = 140
    
    # Calculate scaling factor with adaptive strength
    # More aggressive correction for very dark images, gentle for well-lit images
    if median_brightness < 50:
        # Very dark - strong correction needed
        scale = (target_brightness / max(median_brightness, 1)) ** 0.7
    elif median_brightness < 100:
        # Underexposed - moderate correction
        scale = (target_brightness / max(median_brightness, 1)) ** 0.5
    elif median_brightness > 180:
        # Overexposed - gentle reduction
        scale = (target_brightness / median_brightness) ** 0.3
    else:
        # Well exposed - minimal adjustment
        scale = (target_brightness / median_brightness) ** 0.4
    
    # Apply scaling
    normalized = rgb_float * scale
    
    # Color cast correction (minimal)
    median_r = np.median(normalized[:, 0])
    median_g = np.median(normalized[:, 1])
    median_b = np.median(normalized[:, 2])
    overall_median = (median_r + median_g + median_b) / 3
    
    # Only correct significant color casts
    if overall_median > 10:  # Avoid division by near-zero
        if abs(median_r - overall_median) > overall_median * 0.15:
            scale_r = 1.0 + (overall_median / max(median_r, 1) - 1.0) * 0.3
            normalized[:, 0] *= scale_r
        if abs(median_g - overall_median) > overall_median * 0.15:
            scale_g = 1.0 + (overall_median / max(median_g, 1) - 1.0) * 0.3
            normalized[:, 1] *= scale_g
        if abs(median_b - overall_median) > overall_median * 0.15:
            scale_b = 1.0 + (overall_median / max(median_b, 1) - 1.0) * 0.3
            normalized[:, 2] *= scale_b
    
    return np.clip(normalized, 0, 255)

def find_closest_mst(lab_color):
    """Find closest MST using weighted Delta E with emphasis on color over brightness"""
    L, a, b = lab_color[0, 0]
    
    distances = []
    for ref_lab in mst_lab_palette:
        L_ref, a_ref, b_ref = ref_lab[0, 0]
        
        # Weighted distance: color (a, b) is 2x more important than lightness (L)
        # This helps distinguish skin tones even under varying lighting
        delta_L = (L - L_ref) * 0.5  # Reduced weight for lightness
        delta_a = (a - a_ref) * 1.0  # Full weight for red-green
        delta_b = (b - b_ref) * 1.0  # Full weight for yellow-blue
        
        delta_e = np.sqrt(delta_L**2 + delta_a**2 + delta_b**2)
        distances.append(delta_e)
    
    return distances.index(min(distances)) + 1

def extract_face_skin_color(image_path, model):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    results = model([img_rgb], conf=0.001, imgsz=1280, verbose=False)
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        print(f"No face detected in {image_path}")
        return None

    x1, y1, x2, y2 = map(int, boxes[0].xyxy[0].cpu().numpy())
    face_crop = img_rgb[y1:y2, x1:x2]
    if face_crop.size == 0:
        print(f"Empty face crop in {image_path}")
        return None

    mp_face = mp.solutions.face_mesh
    with mp_face.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True) as face_mesh:
        result = face_mesh.process(face_crop)
        if not result.multi_face_landmarks:
            print(f"No landmarks found in {image_path}")
            return None

        h, w, _ = face_crop.shape
        points = result.multi_face_landmarks[0].landmark
        
        # SAME LANDMARK SET
        # 110, 50, 205 Right cheek
        # 330, 280, 435 Left cheek     
        # 10, 109, 338 forhead
        idxs = [110, 50, 205, 330, 280, 435, 10, 109, 338]

        pixels = []
        sampled_points = []
        pixel_positions = []

        # THE ONLY CHANGE: we sample raw RGB values directly.
        for idx, point in enumerate(points):
            x = min(w - 1, max(0, int(point.x * w)))
            y = min(h - 1, max(0, int(point.y * h)))

            if idx in idxs:
                # 5×5 patch (exactly like before)
                for dy in range(-2, 3):
                    for dx in range(-2, 3):
                        px = min(w - 1, max(0, x + dx))
                        py = min(h - 1, max(0, y + dy))

                        pixel = face_crop[py, px]  # RAW RGB
                        pixels.append(pixel)
                        pixel_positions.append((px, py))

                sampled_points.append((x, y, idx))

        # NO filtering or brightness checks
        if len(pixels) < 10:
            print(f"Not enough raw pixels in {image_path}")
            return None

        pixels = np.array(pixels)

        # RAW mean RGB
        # mean_rgb = np.mean(pixels, axis=0) / 255.0

        # -----------------------------
        # MEDOID instead of MEAN
        # -----------------------------
        # Compute pairwise distances in RGB space
        dists = np.sum((pixels[:, None, :] - pixels[None, :, :])**2, axis=2)
        medoid_idx = np.argmin(np.sum(dists, axis=1))
        medoid_rgb = pixels[medoid_idx] / 255.0
        mean_rgb = medoid_rgb  # use medoid in place of mean
        # -----------------------------


        # Convert to LAB
        lab = rgb2lab([[mean_rgb]])

        ita = compute_ita(lab)
        mst = find_closest_mst(lab)

        subject_id = Path(image_path).parent.name
        gt_mst = actual_annotations.get(subject_id)

        print(f"\nImage: {image_path}")
        print(f"Sampled {len(pixels)} raw pixels from {len(sampled_points)} landmarks")
        print(f"Mean RGB: {mean_rgb * 255}")
        print(f"LAB: L={lab[0,0,0]:.1f}, a={lab[0,0,1]:.1f}, b={lab[0,0,2]:.1f}")
        print(f"Predicted MST: {mst}, Ground Truth: {gt_mst}")

        # DRAW VISUALIZATIONS (unchanged)
        for idx, point in enumerate(points):
            x = min(w - 1, max(0, int(point.x * w)))
            y = min(h - 1, max(0, int(point.y * h)))

            if idx in idxs:
                cv2.circle(face_crop, (x, y), 5, (0, 255, 0), -1)
            else:
                cv2.circle(face_crop, (x, y), 1, (100, 100, 255), -1)

            cv2.putText(face_crop, str(idx), (x, y-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 255), 1)

        # Mark sampled pixels (unchanged)
        for px, py in pixel_positions[:100]:
            cv2.circle(face_crop, (px, py), 1, (255, 255, 0), -1)

        # Draw predicted/GT color boxes (unchanged)
        def draw_color_box(img, top_left, color_rgb, label):
            color_rgb_255 = tuple(int(c * 255) for c in color_rgb)
            x, y = top_left
            cv2.rectangle(img, (x, y), (x + 50, y + 50), color_rgb_255, -1)
            cv2.rectangle(img, (x, y), (x + 50, y + 50), (0, 0, 0), 2)
            cv2.putText(img, label, (x, y - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

        draw_color_box(face_crop, (0, 0), mst_rgb_palette[mst - 1], f"Pred: {mst}")
        if gt_mst:
            draw_color_box(face_crop, (60, 0), mst_rgb_palette[gt_mst - 1], f"GT: {gt_mst}")

        debug_path = image_path.replace(".jpg", "_landmarks_raw.jpg") \
                               .replace("subject_", "landmark_subject_")

        os.makedirs(os.path.dirname(debug_path), exist_ok=True)
        cv2.imwrite(debug_path, cv2.cvtColor(face_crop, cv2.COLOR_RGB2BGR))

        # RETURN FORMAT EXACTLY THE SAME
        return {
            "r": int(mean_rgb[0] * 255),
            "g": int(mean_rgb[1] * 255),
            "b": int(mean_rgb[2] * 255),
            "l": lab[0, 0, 0],
            "a": lab[0, 0, 1],
            "b_lab": lab[0, 0, 2],
            "ita": ita,
            "mst": mst,
            "gt_mst": gt_mst if gt_mst else -1,
            "num_pixels": len(pixels)
        }


def batch_process_images(img_dir, output_csv, model_path="yolov12l-face.pt"):
    model = YOLO(model_path)
    model.fuse()
    records = []

    for path in Path(img_dir).rglob("*.jpg"):
        result = extract_face_skin_color(str(path), model)
        if result:
            result["id"] = path.name
            records.append(result)

    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    
    # Calculate accuracy if ground truth available
    if 'gt_mst' in df.columns:
        valid_preds = df[df['gt_mst'] != -1]
        if len(valid_preds) > 0:
            accuracy = (valid_preds['mst'] == valid_preds['gt_mst']).mean()
            mae = (valid_preds['mst'] - valid_preds['gt_mst']).abs().mean()
            print(f"\nAccuracy: {accuracy:.2%}")
            print(f"Mean Absolute Error: {mae:.2f} MST levels")
    
    print(f"Saved skin tone data for {len(df)} faces to {output_csv}")

# Example usage:
for i in range(19):
    subject = f'subject_{i}'
    print(f"\n{'='*60}")
    print(f"Processing {subject}")
    print(f"{'='*60}")
    batch_process_images(rf"MonkSkinToneDataset\mst-e_data\{subject}", rf"MonkSkinToneDataset\{subject}.csv")

KeyboardInterrupt: 

## Testing Paper Methods

### Loading FACET Dataset
- Link: https://ai.meta.com/datasets/facet-downloads/

In [ ]:
pip uninstall fiftyone

In [17]:
import pandas as pd
def clean_facet_skintone_annotations(annotations_csv, consensus_threshold=0.5):
    gt_df = pd.read_csv(annotations_csv)

    skin_cols = [c for c in gt_df.columns if c.startswith("skin_tone_")]

    # Keep rows where at least one skin tone column is > 0
    gt_df_filtered = gt_df[gt_df[skin_cols].sum(axis=1) > 0]

    print("Original:", len(gt_df))

    # highest vote / total votes >= 0.7 (or whatever threshold you want)
    max_vote = gt_df_filtered[skin_cols].max(axis=1)
    total_vote = gt_df_filtered[skin_cols].sum(axis=1)

    gt_df_consensus = gt_df_filtered[(max_vote / total_vote) >= consensus_threshold]

    # remove rows where skin_tone_na has the highest vote
    gt_df_consensus = gt_df_consensus[gt_df_consensus[skin_cols].idxmax(axis=1) != "skin_tone_na"]

    # Add final MST label (majority vote)
    gt_df_consensus["mst_label"] = gt_df_consensus[skin_cols].idxmax(axis=1).str.replace("skin_tone_", "").astype(int)

    print("Original:", len(gt_df), "Filtered:", len(gt_df_consensus))

    print("MST Label Distribution:")
    print(gt_df_consensus['mst_label'].value_counts().sort_index())
    return gt_df_consensus

annotations_csv = r"G:\Thesis\FACET_Dataset\Annotations\annotations\annotations.csv"
facet_annotations_df = clean_facet_skintone_annotations(annotations_csv, consensus_threshold=0.5)
# display(facet_annotations_df.head())

Original: 49551
Original: 49551 Filtered: 18636
MST Label Distribution:
mst_label
1      318
2     3897
3     3880
4     3348
5     2440
6     2593
7     1046
8      519
9      475
10     120
Name: count, dtype: int64


In [18]:
facet_annotations_df = facet_annotations_df[~facet_annotations_df["mst_label"].isin([2,3,4,5,6,7])]
print("MST Label Distribution:")
print(facet_annotations_df['mst_label'].value_counts().sort_index())

MST Label Distribution:
mst_label
1     318
8     519
9     475
10    120
Name: count, dtype: int64


In [19]:
import os
import cv2
import mediapipe as mp
import numpy as np
from tqdm import tqdm

def segment_and_save(row, face_mesh, input_image_dir, output_image_dir, filename_col='filename'):
    # Assuming your DF has a column 'filename' or similar. Update accordingly.
    # FACET often uses image_id, so you might need: f"{row['image_id']}.jpg"
    image_name = row[filename_col] 
    input_path = os.path.join(input_image_dir, image_name)
    output_path = os.path.join(output_image_dir, image_name)
    
    if os.path.exists(output_path): return output_path

    image = cv2.imread(input_path)
    if image is None: return None

    # 1. MediaPipe Processing
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_image)

    if not results.multi_face_landmarks:
        return None 

    h, w, _ = image.shape
    mask = np.zeros((h, w), dtype=np.uint8)

    # 2. Create the Mask
    for face_landmarks in results.multi_face_landmarks:
        points = [(int(lm.x * w), int(lm.y * h)) for lm in face_landmarks.landmark]
        hull = cv2.convexHull(np.array(points, dtype=np.int32))
        cv2.fillConvexPoly(mask, hull, 255)

    # 3. Apply Mask (Black out background)
    segmented = cv2.bitwise_and(image, image, mask=mask)

    # --- NEW STEP: CROP TO CONTENT ---
    # Find the bounding box of the white area in the mask
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        x, y, w_box, h_box = cv2.boundingRect(contours[0])
        
        # Add a tiny padding if desired (optional, e.g., 10 pixels) so face isn't touching edges
        pad = 10
        x = max(0, x - pad)
        y = max(0, y - pad)
        w_box = min(w, w_box + 2*pad)
        h_box = min(h, h_box + 2*pad)

        # Crop the segmented image
        segmented = segmented[y:y+h_box, x:x+w_box]
    # ---------------------------------

    cv2.imwrite(output_path, segmented)
    return output_path

# Setup MediaPipe
max_num_faces = 1
confidence_threshold = 0 #0.25
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=max_num_faces, refine_landmarks=True, min_detection_confidence=confidence_threshold)

# Define paths
input_image_dir = r'G:\Thesis\FACET_Dataset\Images'
output_image_dir = rf'G:\Thesis\FACET_Dataset\Segmented_Images_{confidence_threshold}'
os.makedirs(output_image_dir, exist_ok=True)

# Run on your dataframe
print("Starting segmentation...")
valid_paths = []
for index, row in tqdm(facet_annotations_df.iterrows(), total=len(facet_annotations_df)):
    path = segment_and_save(row, face_mesh, input_image_dir, output_image_dir, filename_col='filename')
    valid_paths.append(path)

# Update DataFrame to point to new images and drop failures
facet_annotations_df['segmented_path'] = valid_paths
facet_annotations_df = facet_annotations_df.dropna(subset=['segmented_path'])
print(f"Segmentation complete. Usable images: {len(facet_annotations_df)}")

Starting segmentation...


  0%|          | 0/1432 [00:00<?, ?it/s]

100%|██████████| 1432/1432 [01:12<00:00, 19.81it/s]

Segmentation complete. Usable images: 98


In [20]:
print("MST Label Distribution:")
print(facet_annotations_df['mst_label'].value_counts().sort_index())

MST Label Distribution:
mst_label
1     22
8     45
9     25
10     6
Name: count, dtype: int64


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

def apply_gray_world(img):
    """
    Simple 'Gray World' White Balancing.
    Assumes the average color of the scene should be neutral gray.
    """
    if img is None: return None
    result = img.transpose(2, 0, 1).astype(np.float32) # Move channels first
    mean_b = np.mean(result[0])
    mean_g = np.mean(result[1])
    mean_r = np.mean(result[2])
    
    # Avoid division by zero
    mean_gray = (mean_b + mean_g + mean_r) / 3.0 + 1e-6
    
    # Scale channels
    result[0] /= (mean_b / mean_gray)
    result[1] /= (mean_g / mean_gray)
    result[2] /= (mean_r / mean_gray)
    
    result = result.transpose(1, 2, 0).clip(0, 255).astype(np.uint8)
    return result

def extract_features(image_path):
    img = cv2.imread(image_path)
    if img is None: return None
    
    # APPLY WHITE BALANCE
    img = apply_gray_world(img)

    # Create a mask for non-black pixels (since we already blackened the background)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    
    features = []
    
    # Helper for normalized histograms
    def get_hist(image_data, channel):
        hist = cv2.calcHist([image_data], [channel], mask, [256], [0, 256])
        cv2.normalize(hist, hist)
        return hist.flatten()

    # Color Space Conversions
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_ycbcr = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    img_lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

    # Append Features (RGB + Y + V + L)
    for i in range(3): features.extend(get_hist(img_rgb, i)) # R, G, B
    features.extend(get_hist(img_ycbcr, 0)) # Y
    features.extend(get_hist(img_hsv, 2))   # V
    features.extend(get_hist(img_lab, 0))   # L
    
    return np.array(features)

# # Extract for all
# print("Extracting features...")
# X = []
# y = []

# for index, row in tqdm(facet_annotations_df.iterrows(), total=len(facet_annotations_df)):
#     feat = extract_features(row['segmented_path'])
#     if feat is not None:
#         X.append(feat)
#         y.append(row['mst_label'])

# X = np.array(X)
# y = np.array(y)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

# 1. Split out the Test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)

# 2. Split the remaining 85% into Train (approx 70% total) and Validation (approx 15% total)
# 0.176 of 0.85 is roughly 0.15
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=42)

# Train Random Forest (No class weights, as per paper)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced_subsample')
rf_model.fit(X_train, y_train)

def calculate_ooacc(y_true, y_pred):
    """
    Calculates Off-by-One Accuracy.
    Returns the percentage of predictions within distance 1 of the true label.
    """
    # Convert to numpy arrays to ensure element-wise math works
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Calculate absolute difference
    diff = np.abs(y_true - y_pred)
    
    # Check if difference is less than or equal to 1
    # Returns a boolean array, mean() converts it to percentage
    return np.mean(diff <= 1)

# --- Updated Evaluation Block ---

# 1. Get Predictions
val_preds = rf_model.predict(X_val)

# 2. Standard Accuracy (Exact Match)
acc = accuracy_score(y_val, val_preds)

# 3. Off-by-One Accuracy (Metric from the paper)
oo_acc = calculate_ooacc(y_val, val_preds)

print(f"Validation Accuracy:       {acc:.4f}")
print(f"Validation Off-by-One Acc: {oo_acc:.4f}")

# ONLY use X_test at the very end when you are done tuning
# Evaluate
# preds = rf_model.predict(X_test)
# print("CCV Accuracy:", accuracy_score(y_test, preds))
# print(classification_report(y_test, preds))

Validation Accuracy: 0.3302325581395349


In [7]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)

# Train Random Forest (No class weights, as per paper)
rf_model = RandomForestClassifier(n_estimators=1000, random_state=42, class_weight='balanced_subsample', oob_score=True)
rf_model.fit(X_train, y_train)

# Evaluate
preds = rf_model.predict(X_test)
print("CCV Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

CCV Accuracy: 0.36574074074074076
              precision    recall  f1-score   support

           1       0.00      0.00      0.00         3
           2       0.41      0.44      0.43        43
           3       0.34      0.60      0.43        55
           4       0.33      0.24      0.28        38
           5       0.46      0.21      0.29        28
           6       0.37      0.27      0.31        26
           7       0.40      0.17      0.24        12
           8       0.40      0.33      0.36         6
           9       0.50      0.25      0.33         4
          10       0.00      0.00      0.00         1

    accuracy                           0.37       216
   macro avg       0.32      0.25      0.27       216
weighted avg       0.37      0.37      0.35       216



c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels wi

In [30]:
import os
import cv2
import pandas as pd
import numpy as np
import mediapipe as mp
import gc
from tqdm import tqdm

# --- CONFIGURATION ---
MST_E_ROOT = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\MonkSkinToneDataset\mst-e_data'
CSV_PATH = os.path.join(MST_E_ROOT, 'mst-e_image_details.csv')
OUTPUT_DIR = os.path.join(r"G:\Thesis\FACET_Dataset", 'MSTE_Segmented_Images')

# Create output dir
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 1. LOAD AND PREPARE DATAFRAME ---
mst_df = pd.read_csv(CSV_PATH)

# Optional: Filter out masked faces or poor lighting if desired?
# For now, we will keep them and let the geometric filter decide, 
# but usually, you might want to drop 'mask' == 1
# df = df[df['mask'] == 0] 

# Rename 'MST' to 'mst_label' to match FACET schema
mst_df = mst_df.rename(columns={'MST': 'mst_label'})

print(f"Loaded {len(mst_df)} entries from MST-E.")

# --- 2. DEFINE PROCESSING FUNCTION ---
def process_mst_e_image(row, face_mesh, output_dir):
    # Construct paths based on MST-E structure: root/subject_name/image_ID
    # row.subject_name might be "subject_18", row.image_ID is "PXL...jpg"
    
    sub_folder = str(row.subject_name)
    fname = str(row.image_ID)
    
    # Input Path
    input_path = os.path.join(MST_E_ROOT, sub_folder, fname)
    
    # Output Path (We flatten the structure for the segmented folder to make loading easier)
    # New name: subject_18_PXL...jpg to avoid collisions
    new_filename = f"{sub_folder}_{fname}"
    output_path = os.path.join(output_dir, new_filename)
    
    # Skip if already done
    if os.path.exists(output_path):
        return output_path

    # Check if input exists
    if not os.path.exists(input_path):
        return None

    try:
        image = cv2.imread(input_path)
        if image is None: return None
        h_img, w_img, _ = image.shape

        # MediaPipe Processing
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb_image)

        if not results.multi_face_landmarks:
            return None 

        landmarks = results.multi_face_landmarks[0]
        
        # --- GEOMETRIC FILTERS (Same as before) ---
        points = np.array([(int(lm.x * w_img), int(lm.y * h_img)) for lm in landmarks.landmark])
        x, y, w_box, h_box = cv2.boundingRect(points)
        
        # Filter: Resolution & Aspect Ratio
        if w_box < 50 or h_box < 50: return None
        aspect_ratio = h_box / w_box
        if aspect_ratio < 0.6 or aspect_ratio > 2.5: return None
        
        # --- MASKING & CROPPING ---
        mask = np.zeros((h_img, w_img), dtype=np.uint8)
        hull = cv2.convexHull(points)
        cv2.fillConvexPoly(mask, hull, 255)
        
        segmented = cv2.bitwise_and(image, image, mask=mask)
        
        # Crop with padding
        pad = 10
        x = max(0, x - pad)
        y = max(0, y - pad)
        w_box = min(w_img, x + w_box + 2*pad) - x
        h_box = min(h_img, y + h_box + 2*pad) - y
        
        cropped = segmented[y:y+h_box, x:x+w_box]
        
        # Save
        cv2.imwrite(output_path, cropped)
        return output_path

    except Exception as e:
        print(f"Error on {fname}: {e}")
        return None
    finally:
        del image
        if 'rgb_image' in locals(): del rgb_image
        if 'segmented' in locals(): del segmented

# --- 3. RUN PIPELINE ---
mp_face_mesh = mp.solutions.face_mesh
valid_paths = []

print("Starting MST-E Processing...")

with mp_face_mesh.FaceMesh(
    static_image_mode=True, 
    max_num_faces=1, 
    refine_landmarks=True, 
    min_detection_confidence=0.5
) as face_mesh:
    
    for i, row in tqdm(enumerate(df.itertuples(index=False)), total=len(df)):
        path = process_mst_e_image(row, face_mesh, OUTPUT_DIR)
        valid_paths.append(path)
        
        if i % 200 == 0: gc.collect()

# --- 4. SAVE RESULTING DATAFRAME ---
mst_df['segmented_path'] = valid_paths
mst_df = mst_df.dropna(subset=['segmented_path'])

# Save this processed CSV so you can load it easily for training
output_csv_path = os.path.join(MST_E_ROOT, 'mst_e_processed.csv')
mst_df.to_csv(output_csv_path, index=False)

print(f"Done! {len(mst_df)} images successfully processed.")
print(f"Processed CSV saved to: {output_csv_path}")

Loaded 1546 entries from MST-E.
Starting MST-E Processing...


100%|██████████| 1546/1546 [00:09<00:00, 161.96it/s]

Done! 1388 images successfully processed.
Processed CSV saved to: C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\MonkSkinToneDataset\mst-e_data\mst_e_processed.csv


In [31]:
def apply_gray_world_masked(img):
    """
    Applies Gray World White Balancing ONLY on the face pixels.
    Ignores the black background to prevent color skew.
    """
    if img is None: return None
    
    # 1. Create a mask to identify non-black pixels (the face)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > 1  # Pixels that are not black
    
    # If image is empty or all black, return as is
    if np.sum(mask) == 0: 
        return img

    # 2. Calculate average color of the FACE ONLY
    # We use boolean indexing [mask] to select only valid pixels
    mean_b = np.mean(img[:,:,0][mask])
    mean_g = np.mean(img[:,:,1][mask])
    mean_r = np.mean(img[:,:,2][mask])
    
    # 3. Calculate the "Gray" target
    global_mean = (mean_b + mean_g + mean_r) / 3.0
    
    # 4. Calculate scaling factors
    # (Avoid division by zero with 1e-6)
    scale_b = global_mean / (mean_b + 1e-6)
    scale_g = global_mean / (mean_g + 1e-6)
    scale_r = global_mean / (mean_r + 1e-6)
    
    # 5. Apply scaling to the whole image
    # Note: Black pixels (0) * scale will remain 0, so background stays black.
    result = img.astype(np.float32)
    result[:,:,0] *= scale_b
    result[:,:,1] *= scale_g
    result[:,:,2] *= scale_r
    
    return np.clip(result, 0, 255).astype(np.uint8)

def extract_features(image_path):
    """
    Loads image, applies White Balance, and extracts color histograms.
    """
    img = cv2.imread(image_path)
    if img is None: return None
    
    # --- STEP 1: APPLY WHITE BALANCE ---
    img = apply_gray_world_masked(img)
    # -----------------------------------
    
    # Create mask for non-black pixels (re-calculate on balanced image)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    
    features = []
    
    # Helper for normalized histograms
    def get_hist(image_data, channel):
        # Calculate hist only for the masked region
        hist = cv2.calcHist([image_data], [channel], mask, [256], [0, 256])
        cv2.normalize(hist, hist)
        return hist.flatten()

    # Color Space Conversions
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_ycbcr = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    img_lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

    # Extract Features (RGB + Y + V + L) - Same as paper
    for i in range(3): features.extend(get_hist(img_rgb, i)) # R, G, B
    features.extend(get_hist(img_ycbcr, 0)) # Y (Luminance)
    features.extend(get_hist(img_hsv, 2))   # V (Value)
    features.extend(get_hist(img_lab, 0))   # L (Lightness)
    
    return np.array(features)

In [32]:
from tqdm import tqdm
import numpy as np
import pandas as pd

# 3. Bulk Extraction
print("Extracting MST-E features for Random Forest...")

X = []
y = []
subjects = []

for row in tqdm(mst_df.itertuples(index=False), total=len(mst_df)):
    feat = extract_features(row.segmented_path)
    
    # Only append if feature extraction was successful
    if feat is not None:
        X.append(feat)
        y.append(row.mst_label)
        subjects.append(row.subject_name)

# Convert to Arrays
X = np.array(X)
y = np.array(y)
subjects = np.array(subjects)

print("Feature extraction complete!")
print(f"Feature Matrix Shape: {X.shape}")
print(f"Subjects Array Shape: {subjects.shape}") # Should match X[0]

Extracting MST-E features for Random Forest...


100%|██████████| 1388/1388 [01:07<00:00, 20.43it/s]

Feature extraction complete!
Feature Matrix Shape: (1388, 1536)
Subjects Array Shape: (1388,)


In [33]:
import numpy as np
import pandas as pd

# --- 1. Setup Data & Metadata ---
# Create a helper DataFrame to map indices to subjects and labels
split_df = pd.DataFrame({
    'index': np.arange(len(y)), # derived from your y_combined length
    'label': y,
    'subject': subjects
})

train_indices = []
test_indices = []

unique_labels = split_df['label'].unique()

print(f"Splitting data across {len(unique_labels)} skin tones (Train/Test Only)...")

for label in unique_labels:
    # Get all unique subjects for this specific skin tone
    label_subjects = split_df[split_df['label'] == label]['subject'].unique()
    n_subj = len(label_subjects)
    
    # Shuffle subjects to ensure random assignment every run
    np.random.seed(42)
    np.random.shuffle(label_subjects)
    
    # --- LOGIC: Handle Small Subject Counts ---
    
    if n_subj == 1:
        # Case: Only 1 subject. Must go to Train so model learns the class.
        print(f"  - Label {label}: Only 1 subject. Forcing to TRAIN.")
        test_subjs = []
        train_subjs = label_subjects
        
    else:
        # Case: 2 or more subjects.
        # We want roughly 20% in Test, but AT LEAST 1 subject.
        n_test = int(n_subj * 0.20)
        
        # If 20% results in 0 (e.g., 2 subjects * 0.2 = 0.4), force it to 1
        if n_test == 0: 
            n_test = 1
            
        # Split the list
        test_subjs = label_subjects[:n_test]
        train_subjs = label_subjects[n_test:]

    # --- Get Row Indices ---
    # Find the row indices corresponding to these subjects
    train_indices.extend(split_df[split_df['subject'].isin(train_subjs)].index.values)
    test_indices.extend(split_df[split_df['subject'].isin(test_subjs)].index.values)

# --- 2. Create Final Arrays ---
X_train = X[train_indices]
y_train = y[train_indices]

X_test = X[test_indices]
y_test = y[test_indices]

# --- 3. Summary & Verification ---
print("\n" + "="*30)
print(f"Final Split Summary")
print("="*30)
print(f"Train Set: {len(X_train)} images")
print(f"Test Set:  {len(X_test)} images")

# Check Subject Leakage
train_subs_set = set(split_df.iloc[train_indices]['subject'])
test_subs_set = set(split_df.iloc[test_indices]['subject'])
leakage = train_subs_set.intersection(test_subs_set)

if not leakage:
    print("SUCCESS: No subject overlap between Train and Test.")
else:
    print(f"CRITICAL WARNING: Subject leakage detected! {leakage}")

# --- 4. Train Model ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("\nTraining Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced_subsample')
rf_model.fit(X_train, y_train)

# Predict
preds = rf_model.predict(X_test)

# Evaluate
print(f"Test Accuracy: {accuracy_score(y_test, preds):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, preds))

Splitting data across 10 skin tones (Train/Test Only)...
  - Label 10: Only 1 subject. Forcing to TRAIN.
  - Label 7: Only 1 subject. Forcing to TRAIN.

Final Split Summary
Train Set: 821 images
Test Set:  567 images
SUCCESS: No subject overlap between Train and Test.

Training Random Forest...
Test Accuracy: 0.1252

Classification Report:
              precision    recall  f1-score   support

           1       0.48      0.35      0.41        82
           2       0.25      0.45      0.32        56
           3       0.00      0.00      0.00        38
           4       0.10      0.09      0.09        93
           5       0.04      0.04      0.04        73
           6       0.02      0.01      0.01        82
           7       0.00      0.00      0.00         0
           8       0.05      0.03      0.03        73
           9       0.16      0.04      0.07        70
          10       0.00      0.00      0.00         0

    accuracy                           0.13       567
   macro

c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples.

In [34]:
# ... [After your existing code] ...

# --- 5. Per-Subject Detailed Analysis ---
print("\n" + "="*40)
print("PER-SUBJECT PREDICTION BREAKDOWN")
print("="*40)

# 1. Get the list of subjects corresponding to the Test set
test_subjects_arr = split_df.iloc[test_indices]['subject'].values

# 2. Create a temporary DataFrame to aggregate results
results_df = pd.DataFrame({
    'Subject': test_subjects_arr,
    'True_Label': y_test,
    'Predicted': preds
})

# 3. Group by Subject and calculate metrics
subject_stats = []

for subject, group in results_df.groupby('Subject'):
    true_label = group['True_Label'].iloc[0] # The true label for this person
    total_imgs = len(group)
    
    # Calculate exact accuracy for this person
    accuracy = np.mean(group['True_Label'] == group['Predicted'])
    
    # Calculate Off-by-One Accuracy for this person
    off_by_one = np.mean(np.abs(group['True_Label'] - group['Predicted']) <= 1)
    
    # Get the distribution of predictions (e.g., {3: 45, 4: 5})
    # This helps you see if the model is confused between two neighbors
    pred_counts = group['Predicted'].value_counts().sort_index().to_dict()
    
    subject_stats.append({
        'Subject': subject,
        'True_Label': true_label,
        'Images': total_imgs,
        'Accuracy': f"{accuracy:.2%}",
        'Off-By-1': f"{off_by_one:.2%}",
        'Predictions': pred_counts
    })

# 4. Convert to DataFrame for pretty printing
stats_df = pd.DataFrame(subject_stats)

# Sort by True Label so you can see performance across the skin tone spectrum
stats_df = stats_df.sort_values(by='True_Label')

# Display
# Pandas option to show all columns and wider text for the dictionary
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 1000)

print(stats_df)

# Optional: Identify the "Hardest" subjects (0% Accuracy)
print("\n--- Subjects with 0% Accuracy ---")
failures = stats_df[stats_df['Accuracy'] == '0.00%']
if not failures.empty:
    print(failures[['Subject', 'True_Label', 'Predictions']])
else:
    print("None! Every subject had at least some correct predictions.")


PER-SUBJECT PREDICTION BREAKDOWN
      Subject  True_Label  Images Accuracy Off-By-1                                                      Predictions
2  subject_16           1      82   35.37%   47.56%                         {1: 29, 2: 10, 3: 6, 5: 4, 6: 28, 10: 5}
1  subject_13           2      56   44.64%   50.00%                    {2: 25, 3: 3, 4: 18, 5: 2, 7: 2, 8: 3, 10: 3}
0   subject_0           3      38    0.00%   52.63%                     {1: 6, 2: 7, 4: 13, 5: 2, 6: 2, 8: 7, 10: 1}
7   subject_7           4      93    8.60%   52.69%             {1: 1, 2: 9, 4: 8, 5: 41, 6: 12, 7: 2, 8: 1, 10: 19}
6   subject_6           5      73    4.11%   58.90%                          {1: 10, 2: 13, 3: 7, 4: 33, 5: 3, 6: 7}
4   subject_3           6      82    1.22%   25.61%             {1: 6, 2: 24, 4: 3, 5: 11, 6: 1, 7: 9, 8: 25, 10: 3}
3   subject_2           8      73    2.74%   30.14%  {1: 1, 2: 8, 3: 3, 4: 1, 5: 2, 6: 4, 7: 4, 8: 2, 9: 16, 10: 32}
5   subject_4           9     

In [35]:
import os
import cv2
import gc
import random
import numpy as np
import pandas as pd
import mediapipe as mp
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Update this path to your MST-E dataset folder
MST_E_ROOT = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\MonkSkinToneDataset\mst-e_data'
CSV_PATH = os.path.join(MST_E_ROOT, 'mst-e_image_details.csv')
PROCESSED_CSV_PATH = os.path.join(MST_E_ROOT, 'mst_e_processed_robust.csv')
OUTPUT_SEG_DIR = os.path.join(r"G:\Thesis\FACET_Dataset", 'MSTE_Segmented_Images')

# Create directories
os.makedirs(OUTPUT_SEG_DIR, exist_ok=True)

# ==========================================
# 2. IMAGE PROCESSING UTILITIES
# ==========================================

def apply_shades_of_gray_masked(img, power=6):
    """
    Applies 'Shades of Gray' Color Constancy (Edge-based White Balance).
    Only calculates estimation on non-black pixels (the face).
    """
    if img is None: return None
    
    # Mask background (black pixels)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > 1
    if np.sum(mask) == 0: return img

    # Get masked pixels
    b = img[:,:,0][mask].astype(np.float32)
    g = img[:,:,1][mask].astype(np.float32)
    r = img[:,:,2][mask].astype(np.float32)

    # Minkowski Norm
    r_norm = np.power(np.mean(np.power(r, power)), 1/power)
    g_norm = np.power(np.mean(np.power(g, power)), 1/power)
    b_norm = np.power(np.mean(np.power(b, power)), 1/power)

    norm_vec = np.sqrt(r_norm**2 + g_norm**2 + b_norm**2)
    if norm_vec == 0: return img
    
    # Calculate Scales
    scale_r = 1.0 / (r_norm + 1e-6)
    scale_g = 1.0 / (g_norm + 1e-6)
    scale_b = 1.0 / (b_norm + 1e-6)

    # Normalize scales
    max_scale = max(scale_r, scale_g, scale_b)
    scale_r /= max_scale
    scale_g /= max_scale
    scale_b /= max_scale
    
    # Apply
    result = img.astype(np.float32)
    result[:,:,0] *= (scale_b * 255)
    result[:,:,1] *= (scale_g * 255)
    result[:,:,2] *= (scale_r * 255)
    
    return np.clip(result, 0, 255).astype(np.uint8)

def extract_robust_features(image_path):
    """
    Extracts purely chromatic features (Color/Hue) and ignores 
    Luminance/Brightness to prevent lighting bias.
    """
    img = cv2.imread(image_path)
    if img is None: return None
    
    # 1. Apply Color Constancy
    img = apply_shades_of_gray_masked(img, power=6)
    
    # Mask background
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    
    features = []
    
    def get_hist(image_data, channel):
        hist = cv2.calcHist([image_data], [channel], mask, [256], [0, 256])
        cv2.normalize(hist, hist)
        return hist.flatten()

    # 2. Convert to Color Spaces
    img_lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    img_ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # 3. Extract ONLY Color Channels (Ignore L, Y, V)
    # Lab: a (Green-Red), b (Blue-Yellow)
    features.extend(get_hist(img_lab, 1))
    features.extend(get_hist(img_lab, 2))
    
    # YCrCb: Cr (Red diff), Cb (Blue diff)
    features.extend(get_hist(img_ycrcb, 1))
    features.extend(get_hist(img_ycrcb, 2))
    
    # HSV: Hue (Pure Color), Saturation (Intensity)
    features.extend(get_hist(img_hsv, 0))
    features.extend(get_hist(img_hsv, 1))

    return np.array(features)

# ==========================================
# 3. SEGMENTATION PIPELINE
# ==========================================

def process_dataset_segmentation():
    """Runs MediaPipe to segment faces and save crops."""
    print("--- STEP 1: SEGMENTATION ---")
    
    # Load raw csv
    df = pd.read_csv(CSV_PATH)
    # df = df[df['mask'] == 0] # Optional: Drop masked faces
    df = df.rename(columns={'MST': 'mst_label'})
    
    mp_face_mesh = mp.solutions.face_mesh
    valid_paths = []
    
    with mp_face_mesh.FaceMesh(
        static_image_mode=True, 
        max_num_faces=1, 
        refine_landmarks=True, 
        min_detection_confidence=0.5
    ) as face_mesh:
        
        for i, row in tqdm(enumerate(df.itertuples(index=False)), total=len(df)):
            
            # --- Path Construction ---
            sub_folder = str(row.subject_name)
            fname = str(row.image_ID)
            input_path = os.path.join(MST_E_ROOT, sub_folder, fname)
            new_filename = f"{sub_folder}_{fname}"
            output_path = os.path.join(OUTPUT_SEG_DIR, new_filename)
            
            # Check existance
            if os.path.exists(output_path):
                valid_paths.append(output_path)
                continue
            
            if not os.path.exists(input_path):
                valid_paths.append(None)
                continue

            try:
                # Load
                image = cv2.imread(input_path)
                if image is None: 
                    valid_paths.append(None); continue
                
                h_img, w_img, _ = image.shape
                
                # MediaPipe
                rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                results = face_mesh.process(rgb_image)
                
                if not results.multi_face_landmarks:
                    valid_paths.append(None); continue
                
                landmarks = results.multi_face_landmarks[0]
                
                # --- Geometric Filters ---
                points = np.array([(int(lm.x * w_img), int(lm.y * h_img)) for lm in landmarks.landmark])
                x, y, w_box, h_box = cv2.boundingRect(points)
                
                if w_box < 50 or h_box < 50: # Too small
                    valid_paths.append(None); continue
                    
                aspect_ratio = h_box / w_box
                if aspect_ratio < 0.6 or aspect_ratio > 2.5: # Weird shape
                    valid_paths.append(None); continue

                # --- Mask & Crop ---
                mask = np.zeros((h_img, w_img), dtype=np.uint8)
                hull = cv2.convexHull(points)
                cv2.fillConvexPoly(mask, hull, 255)
                
                segmented = cv2.bitwise_and(image, image, mask=mask)
                
                pad = 10
                x = max(0, x - pad)
                y = max(0, y - pad)
                w_box = min(w_img, x + w_box + 2*pad) - x
                h_box = min(h_img, y + h_box + 2*pad) - y
                
                cropped = segmented[y:y+h_box, x:x+w_box]
                
                # Save
                cv2.imwrite(output_path, cropped)
                valid_paths.append(output_path)
                
            except Exception as e:
                print(f"Error: {e}")
                valid_paths.append(None)
            finally:
                if i % 100 == 0: gc.collect()
                
    df['segmented_path'] = valid_paths
    df_clean = df.dropna(subset=['segmented_path'])
    df_clean.to_csv(PROCESSED_CSV_PATH, index=False)
    print(f"Segmentation Done. Saved {len(df_clean)} images to {PROCESSED_CSV_PATH}")
    return df_clean

# ==========================================
# 4. FEATURE EXTRACTION & DATA PREP
# ==========================================

def prepare_data(df):
    print("\n--- STEP 2: FEATURE EXTRACTION (ROBUST) ---")
    
    X, y, subjects = [], [], []
    
    for row in tqdm(df.itertuples(index=False), total=len(df)):
        # Extract purely chromatic features
        feat = extract_robust_features(row.segmented_path)
        
        if feat is not None:
            X.append(feat)
            y.append(row.mst_label)
            subjects.append(row.subject_name)
            
    return np.array(X), np.array(y), np.array(subjects)

# ==========================================
# 5. STRICT SUBJECT SPLITTING
# ==========================================

def get_train_test_split(X, y, subjects):
    print("\n--- STEP 3: STRICT SUBJECT SPLIT ---")
    
    # Helper DF
    split_df = pd.DataFrame({'idx': range(len(y)), 'label': y, 'subject': subjects})
    
    train_idxs, test_idxs = [], []
    unique_labels = sorted(split_df['label'].unique())
    
    for label in unique_labels:
        # Get subjects for this skin tone
        subjs = split_df[split_df['label'] == label]['subject'].unique()
        n_subj = len(subjs)
        
        # Shuffle for randomness
        np.random.seed(42)
        np.random.shuffle(subjs)
        
        if n_subj == 1:
            # Singleton -> Train
            train_subjs = subjs
            test_subjs = []
        else:
            # Split roughly 20% to Test
            n_test = int(n_subj * 0.2)
            if n_test == 0: n_test = 1 # Ensure at least 1 test subject if possible
            
            test_subjs = subjs[:n_test]
            train_subjs = subjs[n_test:]
            
        # Get indices
        train_idxs.extend(split_df[split_df['subject'].isin(train_subjs)]['idx'].values)
        test_idxs.extend(split_df[split_df['subject'].isin(test_subjs)]['idx'].values)
        
    X_train, y_train = X[train_idxs], y[train_idxs]
    X_test, y_test = X[test_idxs], y[test_idxs]
    
    # Leakage Check
    train_s = set(split_df.iloc[train_idxs]['subject'])
    test_s = set(split_df.iloc[test_idxs]['subject'])
    intersection = train_s.intersection(test_s)
    
    print(f"Train Images: {len(X_train)} | Test Images: {len(X_test)}")
    if len(intersection) > 0:
        print(f"WARNING: LEAKAGE DETECTED! {intersection}")
    else:
        print("SUCCESS: Zero Subject Leakage.")
        
    return X_train, X_test, y_train, y_test

# ==========================================
# 6. MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    
    # 1. Check if segmentation is already done to save time
    if os.path.exists(PROCESSED_CSV_PATH):
        print(f"Found processed CSV at {PROCESSED_CSV_PATH}. Loading...")
        df_clean = pd.read_csv(PROCESSED_CSV_PATH)
    else:
        df_clean = process_dataset_segmentation()
        
    # 2. Extract Features
    X, y, subjects = prepare_data(df_clean)
    
    # 3. Split
    X_train, X_test, y_train, y_test = get_train_test_split(X, y, subjects)
    
    # 4. Train
    print("\n--- STEP 4: TRAINING ---")
    rf_model = RandomForestClassifier(
        n_estimators=100, 
        random_state=42, 
        class_weight='balanced_subsample'
    )
    rf_model.fit(X_train, y_train)
    
    # 5. Evaluate
    print("\n--- STEP 5: EVALUATION (UNSEEN SUBJECTS) ---")
    preds = rf_model.predict(X_test)
    
    # Calculate Metrics
    acc = accuracy_score(y_test, preds)
    diff = np.abs(y_test - preds)
    oo_acc = np.mean(diff <= 1)
    
    print(f"Test Accuracy:       {acc:.2%}")
    print(f"Off-by-One Accuracy: {oo_acc:.2%}")
    
    print("\nDetailed Report:")
    print(classification_report(y_test, preds))

--- STEP 1: SEGMENTATION ---


100%|██████████| 1546/1546 [00:08<00:00, 173.75it/s]


Segmentation Done. Saved 1388 images to C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\MonkSkinToneDataset\mst-e_data\mst_e_processed_robust.csv

--- STEP 2: FEATURE EXTRACTION (ROBUST) ---


100%|██████████| 1388/1388 [01:45<00:00, 13.16it/s]



--- STEP 3: STRICT SUBJECT SPLIT ---
Train Images: 821 | Test Images: 567
SUCCESS: Zero Subject Leakage.

--- STEP 4: TRAINING ---

--- STEP 5: EVALUATION (UNSEEN SUBJECTS) ---
Test Accuracy:       17.64%
Off-by-One Accuracy: 46.56%

Detailed Report:
              precision    recall  f1-score   support

           1       0.38      0.20      0.26        82
           2       0.19      0.61      0.29        56
           3       0.00      0.00      0.00        38
           4       0.10      0.05      0.07        93
           5       0.27      0.29      0.28        73
           6       0.35      0.24      0.29        82
           7       0.00      0.00      0.00         0
           8       0.09      0.03      0.04        73
           9       0.15      0.03      0.05        70
          10       0.00      0.00      0.00         0

    accuracy                           0.18       567
   macro avg       0.15      0.14      0.13       567
weighted avg       0.21      0.18      0.17 

c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples.

In [19]:
# --- Custom Stratified Group Split ---

# Helper DataFrame to manage indices (we don't put the images here, just metadata)
# We use the arrays we just created
split_df = pd.DataFrame({
    'index': np.arange(len(y)),
    'label': y,
    'subject': subjects
})

train_indices = []
val_indices = []
test_indices = []

unique_labels = split_df['label'].unique()

print(f"Splitting data across {len(unique_labels)} skin tones...")

for label in unique_labels:
    # Get all unique subjects for this specific skin tone
    # We filter by label first, then look at unique subjects
    label_subjects = split_df[split_df['label'] == label]['subject'].unique()
    
    # --- CONSTRAINT: Single Subject Rule ---
    if len(label_subjects) == 1:
        print(f"  - Label {label}: Only 1 subject found ({label_subjects[0]}). Forcing to TRAIN.")
        # Find all indices for this subject and add to train
        subj_indices = split_df[split_df['subject'] == label_subjects[0]].index.values
        train_indices.extend(subj_indices)
        continue

    # --- Standard Split (Approx 70 / 15 / 15) ---
    # Shuffle subjects to ensure random assignment
    np.random.seed(42)
    np.random.shuffle(label_subjects)
    
    n_subj = len(label_subjects)
    n_test = int(n_subj * 0.15)
    n_val = int(n_subj * 0.15)
    
    # Ensure at least 1 subject in test/val if we have enough subjects (e.g. > 3)
    if n_test == 0 and n_subj >= 1: n_test = 1
    if n_val == 0 and n_subj >= 1: n_val = 1
    
    # Slice the subject list
    test_subjs = label_subjects[:n_test]
    val_subjs = label_subjects[n_test : n_test + n_val]
    train_subjs = label_subjects[n_test + n_val:]
    
    # Fallback: If rounding left Train empty but we have subjects, move one back
    if len(train_subjs) == 0 and len(label_subjects) > 1:
        train_subjs = np.append(train_subjs, val_subjs[-1])
        val_subjs = val_subjs[:-1]
        
    # Find the row indices corresponding to these subjects
    train_indices.extend(split_df[split_df['subject'].isin(train_subjs)].index.values)
    val_indices.extend(split_df[split_df['subject'].isin(val_subjs)].index.values)
    test_indices.extend(split_df[split_df['subject'].isin(test_subjs)].index.values)

# --- Create Final Sets ---
X_train = X[train_indices]
y_train = y[train_indices]

X_val = X[val_indices]
y_val = y[val_indices]

X_test = X[test_indices]
y_test = y[test_indices]

print("\n--- Split Summary ---")
print(f"Train: {len(X_train)} images")
print(f"Val:   {len(X_val)} images")
print(f"Test:  {len(X_test)} images")

# VERIFICATION: Check for subject leakage
train_subs_set = set(split_df.iloc[train_indices]['subject'])
val_subs_set = set(split_df.iloc[val_indices]['subject'])
test_subs_set = set(split_df.iloc[test_indices]['subject'])

leakage_tv = train_subs_set.intersection(val_subs_set)
leakage_tt = train_subs_set.intersection(test_subs_set)

if not leakage_tv and not leakage_tt:
    print("\n SUCCESS: No subject overlap between sets.")
else:
    print(f"\n WARNING: Leakage detected! {leakage_tv} {leakage_tt}")

Splitting data across 10 skin tones...
  - Label 10: Only 1 subject found (subject_12). Forcing to TRAIN.
  - Label 7: Only 1 subject found (subject_5). Forcing to TRAIN.

--- Split Summary ---
Train: 738 images
Val:   83 images
Test:  567 images

 SUCCESS: No subject overlap between sets.


In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=1000, random_state=42, class_weight='balanced_subsample')
rf_model.fit(X_train, y_train)

# Predict on Validation (Unseen Subjects)
val_preds = rf_model.predict(X_val)

# Metrics
acc = accuracy_score(y_val, val_preds)
oo_acc = calculate_ooacc(y_val, val_preds) # Ensure your calculate_ooacc function is defined

print(f"Validation Accuracy:       {acc:.4f}")
print(f"Validation Off-by-One Acc: {oo_acc:.4f}")

Validation Accuracy:       0.0000
Validation Off-by-One Acc: 0.0482


In [3]:
# --- ASSUMPTION: You have 'facet_df' (processed) and 'mst_e_df' (processed) ---
# If not, load them:
# facet_df = pd.read_csv("path/to/facet_processed.csv")
# mst_e_df = pd.read_csv("path/to/mst_e_processed.csv")

print("Merging datasets...")
# 1. Combine DataFrames
cols = ['segmented_path', 'mst_label']
full_df = df_clean #pd.concat([facet_df[cols], mst_e_df[cols]], axis=0).reset_index(drop=True)

print(f"Total Combined Images: {len(full_df)}")

# 2. Define Feature Extractor (Same as before)
def extract_features(image_path):
    img = cv2.imread(image_path)
    if img is None: return None
    
    # Create mask for non-black pixels
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    
    features = []
    
    # Helper
    def get_hist(image_data, channel):
        hist = cv2.calcHist([image_data], [channel], mask, [256], [0, 256])
        cv2.normalize(hist, hist)
        return hist.flatten()

    # Color Spaces
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_ycbcr = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    img_lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

    # Extract (RGB + Y + V + L)
    for i in range(3): features.extend(get_hist(img_rgb, i))
    features.extend(get_hist(img_ycbcr, 0)) # Y
    features.extend(get_hist(img_hsv, 2))   # V
    features.extend(get_hist(img_lab, 0))   # L
    
    return np.array(features)

# 3. Bulk Extraction
print("Extracting features for Random Forest...")
X_combined = []
y_combined = []

for row in tqdm(full_df.itertuples(index=False), total=len(full_df)):
    feat = extract_features(row.segmented_path)
    if feat is not None:
        X_combined.append(feat)
        y_combined.append(row.mst_label)

X_combined = np.array(X_combined)
y_combined = np.array(y_combined)

print("Feature extraction complete!")
print(f"Feature Matrix Shape: {X_combined.shape}")

Merging datasets...
Total Combined Images: 1388
Extracting features for Random Forest...


100%|██████████| 1388/1388 [00:38<00:00, 36.06it/s]

Feature extraction complete!
Feature Matrix Shape: (1388, 1536)


In [9]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Split out the Test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(X_combined, y_combined, test_size=0.15, stratify=y_combined, random_state=42)

# 2. Split the remaining 85% into Train (approx 70% total) and Validation (approx 15% total)
# 0.176 of 0.85 is roughly 0.15
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=42)

# Train Random Forest (No class weights, as per paper)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced_subsample')
rf_model.fit(X_train, y_train)

def calculate_ooacc(y_true, y_pred):
    """
    Calculates Off-by-One Accuracy.
    Returns the percentage of predictions within distance 1 of the true label.
    """
    # Convert to numpy arrays to ensure element-wise math works
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Calculate absolute difference
    diff = np.abs(y_true - y_pred)
    
    # Check if difference is less than or equal to 1
    # Returns a boolean array, mean() converts it to percentage
    return np.mean(diff <= 1)

# --- Updated Evaluation Block ---

# 1. Get Predictions
val_preds = rf_model.predict(X_val)

# 2. Standard Accuracy (Exact Match)
acc = accuracy_score(y_val, val_preds)

# 3. Off-by-One Accuracy (Metric from the paper)
oo_acc = calculate_ooacc(y_val, val_preds)

print(f"Validation Accuracy:       {acc:.4f}")
print(f"Validation Off-by-One Acc: {oo_acc:.4f}")

# ONLY use X_test at the very end when you are done tuning
# Evaluate
# preds = rf_model.predict(X_test)
# print("CCV Accuracy:", accuracy_score(y_test, preds))
# print(classification_report(y_test, preds))

Validation Accuracy:       0.7548
Validation Off-by-One Acc: 0.8558


In [37]:
# pip install albumentations

In [38]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import albumentations as A  # <--- NEW LIBRARY

# ==========================================
# 1. CONFIGURATION
# ==========================================
MST_E_ROOT = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\MonkSkinToneDataset\mst-e_data'
MST_E_PROCESSED_CSV = os.path.join(MST_E_ROOT, 'mst_e_processed_robust.csv')

# How many new variations to create per training image?
AUGMENTATION_FACTOR = 10  # If you have 800 train images, you will get ~8800 total.

# ==========================================
# 2. DEFINE AUGMENTATION PIPELINE
# ==========================================
# STRICTLY Geometric and Quality transformations. 
# NO Color/Brightness changes allowed.
augmenter = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=25, p=0.7), # Rotate +/- 25 degrees
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.4), # Simulates camera grain
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),    # Simulates out-of-focus
    A.Affine(scale=(0.8, 1.2), translate_percent=(0.1, 0.1), p=0.5), # Zoom/Shift
    # A.RandomBrightnessContrast(p=0) <--- DO NOT USE THIS
])

# ==========================================
# 3. ROBUST FEATURE EXTRACTOR (Same as before)
# ==========================================
def apply_shades_of_gray_masked(img, power=6):
    if img is None: return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > 1
    if np.sum(mask) == 0: return img
    b = img[:,:,0][mask].astype(np.float32)
    g = img[:,:,1][mask].astype(np.float32)
    r = img[:,:,2][mask].astype(np.float32)
    r_norm = np.power(np.mean(np.power(r, power)), 1/power)
    g_norm = np.power(np.mean(np.power(g, power)), 1/power)
    b_norm = np.power(np.mean(np.power(b, power)), 1/power)
    norm_vec = np.sqrt(r_norm**2 + g_norm**2 + b_norm**2)
    if norm_vec == 0: return img
    scale_r = 1.0 / (r_norm + 1e-6); scale_g = 1.0 / (g_norm + 1e-6); scale_b = 1.0 / (b_norm + 1e-6)
    max_scale = max(scale_r, scale_g, scale_b)
    scale_r /= max_scale; scale_g /= max_scale; scale_b /= max_scale
    result = img.astype(np.float32)
    result[:,:,0] *= (scale_b * 255); result[:,:,1] *= (scale_g * 255); result[:,:,2] *= (scale_r * 255)
    return np.clip(result, 0, 255).astype(np.uint8)

def extract_features_v2(img):
    """
    Accepts an IMAGE ARRAY (not path) so we can pass augmented images directly.
    """
    if img is None: return None
    
    # 1. Apply Color Constancy
    img = apply_shades_of_gray_masked(img, power=6)
    
    # Mask background
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    
    features = []
    
    def get_hist(image_data, channel):
        hist = cv2.calcHist([image_data], [channel], mask, [256], [0, 256])
        cv2.normalize(hist, hist)
        return hist.flatten()

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    img_ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # RGB
    features.extend(get_hist(img_rgb, 0)); features.extend(get_hist(img_rgb, 1)); features.extend(get_hist(img_rgb, 2))
    # Lightness
    features.extend(get_hist(img_lab, 0)); features.extend(get_hist(img_ycrcb, 0)); features.extend(get_hist(img_hsv, 2))
    # Color
    features.extend(get_hist(img_lab, 1)); features.extend(get_hist(img_lab, 2)); features.extend(get_hist(img_hsv, 0))

    return np.array(features)

# ==========================================
# 4. LOAD, SPLIT, THEN AUGMENT
# ==========================================

if __name__ == "__main__":
    print("Loading MST-E CSV...")
    df = pd.read_csv(MST_E_PROCESSED_CSV)
    
    # 1. Perform Subject Split FIRST
    # We must identify which subjects are Train vs Test BEFORE augmentation
    # to prevent leakage.
    split_df = pd.DataFrame({'idx': df.index, 'label': df['mst_label'], 'subject': df['subject_name']})
    train_idxs, test_idxs = [], []
    
    print("Splitting Subjects...")
    for label in sorted(split_df['label'].unique()):
        subjs = split_df[split_df['label'] == label]['subject'].unique()
        np.random.seed(42)
        np.random.shuffle(subjs)
        
        if len(subjs) == 1:
            train_subjs = subjs
            test_subjs = []
        else:
            n_test = max(1, int(len(subjs) * 0.2))
            test_subjs = subjs[:n_test]
            train_subjs = subjs[n_test:]
            
        train_idxs.extend(split_df[split_df['subject'].isin(train_subjs)]['idx'].values)
        test_idxs.extend(split_df[split_df['subject'].isin(test_subjs)]['idx'].values)
        
    df_train = df.loc[train_idxs]
    df_test = df.loc[test_idxs]
    
    print(f"Original Train: {len(df_train)} images")
    print(f"Original Test:  {len(df_test)} images")

    # 2. Process & Augment TRAINING Data
    print(f"\nProcessing TRAIN set with Augmentation (x{AUGMENTATION_FACTOR})...")
    X_train, y_train = [], []
    
    for row in tqdm(df_train.itertuples(index=False), total=len(df_train)):
        if not os.path.exists(row.segmented_path): continue
        
        # Load Original
        original_img = cv2.imread(row.segmented_path)
        if original_img is None: continue
        
        # A. Extract Original
        feat = extract_features_v2(original_img)
        if feat is not None:
            X_train.append(feat)
            y_train.append(row.mst_label)
            
        # B. Generate Augmentations
        for _ in range(AUGMENTATION_FACTOR):
            # Albumentations expects RGB usually, but OpenCV is BGR. 
            # Since our geometric transforms don't care about color order, we can pass BGR.
            # However, noise/blur might behave slightly differently. Safer to convert.
            aug = augmenter(image=original_img)['image']
            
            feat_aug = extract_features_v2(aug)
            if feat_aug is not None:
                X_train.append(feat_aug)
                y_train.append(row.mst_label)

    X_train = np.array(X_train)
    y_train = np.array(y_train)
    print(f"Augmented Train Size: {X_train.shape[0]} samples")

    # 3. Process TEST Data (NO Augmentation)
    print("\nProcessing TEST set (Originals Only)...")
    X_test, y_test = [], []
    
    for row in tqdm(df_test.itertuples(index=False), total=len(df_test)):
        if not os.path.exists(row.segmented_path): continue
        img = cv2.imread(row.segmented_path)
        feat = extract_features_v2(img)
        if feat is not None:
            X_test.append(feat)
            y_test.append(row.mst_label)
            
    X_test = np.array(X_test)
    y_test = np.array(y_test)

    # 4. Train
    print("\nTraining Random Forest on Augmented Data...")
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced_subsample')
    rf_model.fit(X_train, y_train)

    # 5. Evaluate
    print("\n--- EVALUATION (UNSEEN SUBJECTS) ---")
    preds = rf_model.predict(X_test)
    
    acc = accuracy_score(y_test, preds)
    diff = np.abs(y_test - preds)
    oo_acc = np.mean(diff <= 1)
    
    print(f"Test Accuracy:       {acc:.2%}")
    print(f"Off-by-One Accuracy: {oo_acc:.2%}")
    print("\n" + classification_report(y_test, preds, zero_division=0))

c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\User\AppData\Local\Temp\ipykernel_29884\3799414383.py:27: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.4), # Simulates camera grain


Loading MST-E CSV...
Splitting Subjects...
Original Train: 821 images
Original Test:  567 images

Processing TRAIN set with Augmentation (x10)...


100%|██████████| 821/821 [14:48<00:00,  1.08s/it]


Augmented Train Size: 9031 samples

Processing TEST set (Originals Only)...


100%|██████████| 567/567 [00:49<00:00, 11.55it/s]



Training Random Forest on Augmented Data...

--- EVALUATION (UNSEEN SUBJECTS) ---
Test Accuracy:       14.11%
Off-by-One Accuracy: 39.33%

              precision    recall  f1-score   support

           1       0.29      0.18      0.22        82
           2       0.15      0.64      0.24        56
           3       0.00      0.00      0.00        38
           4       0.14      0.11      0.12        93
           5       0.24      0.12      0.16        73
           6       0.32      0.07      0.12        82
           7       0.00      0.00      0.00         0
           8       0.23      0.04      0.07        73
           9       0.04      0.01      0.02        70
          10       0.00      0.00      0.00         0

    accuracy                           0.14       567
   macro avg       0.14      0.12      0.10       567
weighted avg       0.19      0.14      0.13       567

